# GPTWake — Train **XLS-R 300M** (học trước trên 436 nghìn giờ tiếng nói) để phát hiện "Hey GPT"

Notebook này train và đánh giá mô hình **XLS-R 300M** (Babu và cộng sự, 2021 (wav2vec 2.0: Baevski và cộng sự, NeurIPS 2020)) cho bài toán của GPTWake:
nghe một đoạn âm thanh **2,5 giây** và trả lời **có** hay **không** có câu "Hey GPT" trong đó.

Mô hình đọc **thẳng dạng sóng âm (waveform)** như M5 và SincConv: 40 000 con số biên độ (2,5 s × 16 000 mẫu/giây). Khác biệt lớn: phần thân (backbone) của XLS-R **đã được học trước** trên khoảng 436 nghìn giờ tiếng nói không nhãn của 128 ngôn ngữ (gồm cả tiếng Việt và tiếng Anh), bởi Meta AI. Notebook chỉ gắn thêm một đầu phân loại nhỏ và tinh chỉnh (fine-tune) vài lớp trên cùng cho bài toán "Hey GPT". Đây là hướng **học chuyển giao**: mượn hiểu biết về hàng nghìn giọng người thật để bù cho việc dữ liệu train của nhóm phần lớn là giọng TTS.

XLS-R 300M có khoảng **316 triệu tham số**, lớn hơn BC-ResNet-3 khoảng 6 000 lần và tốn khoảng 1 000 lần phép tính cho mỗi cửa sổ. Mục đích của notebook này là trả lời: *học trước trên nhiều giọng người thật có giảm được tỉ lệ bỏ sót giọng người lạ không, và phải trả giá bao nhiêu về tốc độ?*

**Notebook làm trọn một vòng:** đọc dữ liệu → xem/nghe thử → dựng mô hình → train → chọn ngưỡng
trên tập validation → chấm điểm trên tập holdout → đo tốc độ trên CPU → lưu kết quả để so sánh.

---

### Cách chạy trên Kaggle

1. **Mở notebook:** trên Kaggle chọn *Create → New Notebook*, rồi *File → Import Notebook* và chọn file `.ipynb` này.
2. **Gắn dữ liệu:** bảng bên phải chọn *Add Input*, tìm dataset **`cuonghoangquoc/gptwake`** (dataset chính thức của nhóm,
   dạng Parquet) rồi bấm *Add*. Notebook **tự tìm** dữ liệu trong `/kaggle/input`, không cần gõ đường dẫn.
3. **Bật GPU:** *Settings → Accelerator → GPU T4 x2* hoặc *GPU P100*. Notebook dùng một GPU. Không cần Internet.
4. **Chỉnh mục 1 (Cấu hình)** nếu muốn: chọn phân phối train (`H00`, `H25`, `H50`, `H70`), số seed, số epoch…
5. **Chạy:** *Run All* để chạy và xem trực tiếp, hoặc *Save Version → Save & Run All (Commit)* để Kaggle chạy nền
   (đóng trình duyệt cũng không sao). Kết quả nằm trong `/kaggle/working/outputs/`, tải về ở tab *Output*.

**Dữ liệu:** dataset Kaggle `cuonghoangquoc/gptwake` đã có sẵn validation, holdout và cả bốn phân phối train
(dạng Parquet, xem mục 3). **Không ai cần `git lfs pull` toàn bộ dữ liệu về máy:** GitHub tính băng thông LFS vào
tài khoản chủ repo (gói miễn phí khoảng 10 GiB/tháng), một lần tải đủ đã khoảng 6 GB.

**Chạy thử nhanh trước:** đặt `"smoke_test": True` ở mục 1. Notebook chỉ dùng vài trăm file và 2 epoch,
chạy xong trong vài phút. Dùng để kiểm tra dataset gắn đúng chưa, trước khi train thật.

**Chạy trên máy (không phải Kaggle):** tải các file Parquet của dataset về một thư mục rồi ghi đường dẫn vào
`CONFIG["data_root"]`. Hoặc mở notebook trong thư mục `notebooks/` của repo: notebook tự tìm `data/validation`,
`data/holdout` và `data/train/H25` (tạo bằng `python scripts/create_train.py`).

---

### Mục lục

0. Bức tranh tổng thể
1. Cấu hình
2. Thư viện và thiết bị
3. Tìm dữ liệu
4. Đọc âm thanh vào bộ nhớ
5. Xem và nghe thử dữ liệu
6. Mô hình XLS-R 300M
7. Vòng train
8. Cách đánh giá: ngưỡng, FRR, FAR, ROC
9. Chạy thí nghiệm
10. Đường cong học
11. Kết quả trên holdout
12. Nhìn vào bên trong mô hình đã học
13. Đánh đổi độ chính xác – tốc độ (test CPU, batch 1)
14. (Tuỳ chọn) Đo báo động nhầm mỗi giờ trên bản ghi dài
15. Tổng kết và gợi ý thí nghiệm tiếp theo

## 0. Bức tranh tổng thể

### Bài toán

Đầu vào là một **cửa sổ âm thanh 2,5 giây**, mono, 16 kHz. Đầu ra là **một con số** gọi là *logit*.
Đưa logit qua hàm sigmoid được **xác suất** có "Hey GPT" (từ 0 đến 1). Xác suất vượt **ngưỡng** thì coi là có.

```
file WAV 2,5 s ──► 40 000 số nguyên (int16) ──► chia 32 768 ──► 40 000 số thực trong [-1, 1]
                                                                  │
                                         ┌────────────────────────┘
                                         ▼
                                   MÔ HÌNH (waveform → logit)
                                         │
                                  sigmoid(logit) = xác suất "Hey GPT"
                                         │
                              xác suất ≥ ngưỡng ?  ──► có / không
```

Khi chạy thật (streaming), hệ thống cứ mỗi bước nhỏ (ví dụ 0,25 s) lại cắt cửa sổ 2,5 s gần nhất và hỏi mô hình.
Độ dài cửa sổ, bước trượt và ngưỡng đều cấu hình được (quy tắc của repo).

### Ba tập dữ liệu, ba vai trò

| Tập | Dùng để | Ghi chú |
|---|---|---|
| **train** (`H00`/`H25`/`H50`/`H70`) | Mô hình học trọng số | 25 000 positive + 21 400 negative. Tên `Hxx` = xx% negative là *hard negative*. |
| **validation** | Chọn epoch tốt nhất và chọn ngưỡng | 100 positive + 100 negative. Không dùng để học. |
| **holdout** (test) | Chấm điểm **cuối cùng**, chỉ nhìn sau khi đã chốt mọi lựa chọn | 300 positive + 400 negative. Giọng người thật trong holdout (dat, Duc, trieuho2312) **không có** trong train. |

**Lưu ý khi đọc kết quả:** theo manifest của nhóm, chỉ khoảng 5% positive trong train là giọng người thật
(duong, hanhh); phần còn lại là giọng TTS. Holdout có 200 positive là giọng người thật **mới**. Điểm trên holdout vì thế
đo khả năng chuyển từ "chủ yếu TTS" sang giọng người lạ. Người nói của validation (duong, hanhh) lại có bản ghi khác
trong train, nên validation dễ hơn holdout.

**Hard negative** là câu *gần giống* "Hey GPT" nhưng không phải (ví dụ "Hey GP…", "OK GPT", "Hey Siri"). Đây là loại
dễ gây báo nhầm nhất. **Ordinary negative** là lời nói và âm thanh bình thường (vlog, hội thoại…).

**Vì sao không chọn ngưỡng trên holdout?** Nếu vừa chỉnh vừa chấm trên cùng một tập, điểm sẽ đẹp giả tạo.
Holdout giống bài thi cuối kỳ: chỉ làm một lần, sau khi đã ôn xong.

### Hai loại lỗi

- **Bỏ sót (false reject):** người dùng nói "Hey GPT" mà máy không phản ứng. Tỉ lệ gọi là **FRR**.
- **Báo nhầm (false accept / false alarm):** không ai gọi mà máy tự bật. Tỉ lệ gọi là **FAR**.

Ngưỡng cao thì ít báo nhầm nhưng hay bỏ sót, ngưỡng thấp thì ngược lại. Mục 8 giải thích kỹ.

## 1. Cấu hình

Mọi thứ bạn có thể muốn chỉnh đều nằm trong ô dưới. Các ô sau chỉ đọc từ đây.

**Dữ liệu**
- `data_root`: thư mục chứa dữ liệu (`validation.parquet` + `holdout.parquet`, hoặc thư mục `validation/` + `holdout/`). Để `None` thì notebook tự tìm trong `/kaggle/input` (hoặc trong repo khi chạy trên máy).
- `train_splits`: phân phối train, chọn trong `"H00"`, `"H25"`, `"H50"`, `"H70"`. Ghi nhiều tên, ví dụ `["H00", "H50"]`, để chạy lần lượt từng cái.
- `seeds`: seed ngẫu nhiên. Train lại với seed khác cho kết quả hơi khác. Ghi `[0, 1, 2]` để đo độ dao động (nên làm khi viết báo cáo).
- `window_seconds`, `sample_rate`: độ dài cửa sổ và tần số lấy mẫu. Dữ liệu của nhóm là 2,5 s và 16 kHz. File ngắn hơn được đệm số 0, dài hơn bị cắt.

**Train**
- `epochs`: số lần duyệt hết tập train. `batch_size`: số đoạn âm trong một bước cập nhật.
- `learning_rate`: tốc độ học (bước nhảy của mỗi lần cập nhật). `weight_decay`: phạt trọng số lớn, giúp đỡ học vẹt.
- `lr_step_epochs`, `lr_gamma`: cứ `lr_step_epochs` epoch thì nhân tốc độ học với `lr_gamma`. `None` = giữ nguyên.
- `patience`: dừng sớm nếu loss trên validation không giảm sau chừng này epoch liên tiếp. `None` = luôn chạy đủ `epochs`.
- `use_amp`: train bằng số thực 16-bit trên GPU cho nhanh (*mixed precision*). Không ảnh hưởng đáng kể tới kết quả.

**Ngưỡng và đánh giá** (giải thích ở mục 8)
- `threshold_mode`: cách chọn ngưỡng trên validation. `"youden"` = cân bằng hai loại lỗi; `"target_far"` = giữ tỉ lệ báo nhầm ≤ `target_far`; `"fixed"` = dùng `fixed_threshold`.
- `far_points`: các mức FAR để báo cáo "FRR tại FAR cố định".

**Test trên CPU, batch 1** (giải thích ở mục 9)
- `cpu_test`: `True` = chấm holdout bằng CPU, từng clip một, ghi thời gian suy luận của từng clip.
- `cpu_threads`: số luồng CPU khi test. Mặc định `[1, None]`: đo một lần với 1 luồng, một lần với mọi luồng.
- `cpu_warmup`: số lần chạy làm nóng trước khi bấm giờ.
- `checkpoints_to_test`: danh sách đường dẫn `model.pt`. Có thì notebook **không train**, chỉ nạp các checkpoint đó và test.
  Dùng để đo lại thời gian trên laptop của nhóm.

**Streaming (tuỳ chọn, mục 14)**
- `long_negative_dir`: thư mục chứa bản ghi dài *không* có "Hey GPT". Có thì notebook đo số lần báo nhầm mỗi giờ.
- `stream_hop_seconds`: bước trượt cửa sổ. `refractory_seconds`: sau một lần kích hoạt thì im lặng chừng này giây.

**Mô hình**: `MODEL_OPTIONS` ở cuối ô, giải thích ở mục 6.

**Khác**
- `smoke_test`: `True` = chạy thử thật nhanh (vài trăm file, 2 epoch) để kiểm tra mọi thứ nối đúng.
- `output_dir`: nơi lưu kết quả. Trên Kaggle thư mục làm việc là `/kaggle/working`, nên kết quả ở `/kaggle/working/outputs`.

In [ ]:
# ===================== CẤU HÌNH (chỉnh ở đây) =====================
MODEL_NAME = "xlsr"

CONFIG = {
    # --- Dữ liệu ---
    "data_root": None,             # None = tự tìm. Ví dụ: "/kaggle/input/gptwake"
    "train_splits": ["H25"],       # chọn trong "H00", "H25", "H50", "H70"
    "seeds": [0],                  # ví dụ [0, 1, 2] để chạy 3 lần
    "sample_rate": 16_000,
    "window_seconds": 2.5,
    # --- Train ---
    "epochs": 4,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "weight_decay": 1e-2,
    "backbone_lr": 5e-5,           # tốc độ học của phần XLS-R được tinh chỉnh; learning_rate ở trên là của đầu phân loại
    "optimizer": "adamw",          # AdamW: cách thường dùng khi tinh chỉnh Transformer
    "lr_schedule": "cosine",       # warmup rồi giảm theo cosin về 0
    "warmup_epochs": 0.5,
    "eval_batch_size": 64,         # batch khi chấm validation/holdout (mô hình lớn, giữ vừa bộ nhớ GPU)
    "lr_step_epochs": None,
    "lr_gamma": 0.5,
    "patience": None,
    "use_amp": True,
    # --- Ngưỡng và đánh giá ---
    "threshold_mode": "youden",    # "youden" | "target_far" | "fixed"
    "target_far": 0.01,
    "fixed_threshold": 0.5,
    "far_points": [0.005, 0.01, 0.05],
    # --- Test holdout trên CPU, batch 1 ---
    "cpu_test": True,              # chấm holdout trên CPU, từng clip một, ghi thời gian từng clip
    "cpu_threads": [1, None],      # số luồng CPU khi test; None = mọi luồng
    "cpu_warmup": 10,              # số lần chạy làm nóng trước khi bấm giờ
    "cpu_test_clips": 100,         # chỉ bấm giờ CPU trên 100 clip đầu (XLS-R chạy CPU ~1 s/clip); None = cả 700
    "checkpoints_to_test": [],     # có đường dẫn model.pt thì BỎ QUA train, chỉ test (ví dụ đo lại trên laptop)
    # --- Streaming (tuỳ chọn) ---
    "long_negative_dir": None,     # ví dụ "/kaggle/input/gptwake-long-negatives"
    "stream_hop_seconds": 0.25,
    "refractory_seconds": 1.0,
    # --- Khác ---
    "smoke_test": False,
    "smoke_files_per_class": 128,
    "output_dir": "outputs",
}

# Siêu tham số kiến trúc (giải thích ở mục 6). Mặc định = đúng bài báo.
MODEL_OPTIONS = {
    "checkpoint": "facebook/wav2vec2-xls-r-300m",  # tải từ Hugging Face (cần bật Internet), hoặc đường dẫn thư mục
    "finetune_mode": "partial",     # "probe" = đóng băng cả XLS-R; "partial" = tinh chỉnh vài lớp trên; "full" = mọi lớp trừ CNN
    "trainable_top_layers": 4,      # số lớp Transformer trên cùng được tinh chỉnh (khi "partial")
    "use_weighted_layer_sum": True, # trộn đầu ra của mọi lớp với trọng số học được (kiểu SUPERB)
    "projector_dim": 256,
    "classifier_dropout": 0.1,
    "mask_time_prob": None,         # None = giữ giá trị trong checkpoint (che ngẫu nhiên các khung khi train)
    "layerdrop": None,              # None = giữ giá trị trong checkpoint
    "gradient_checkpointing": False,# True nếu thiếu bộ nhớ GPU (chậm hơn)
}

if CONFIG["smoke_test"]:
    CONFIG["epochs"], CONFIG["patience"] = 2, None

## 2. Thư viện và thiết bị

Notebook chỉ cần các thư viện Kaggle cài sẵn: PyTorch, NumPy, SciPy, pandas, scikit-learn, matplotlib.

- `DEVICE` là GPU nếu có, không thì CPU. Train trên CPU được nhưng chậm hơn nhiều.
- `set_seed` đặt hạt giống cho mọi bộ sinh số ngẫu nhiên, để chạy lại cho kết quả gần như y hệt.
  (Trên GPU vẫn có thể lệch rất nhẹ vì một số phép tính song song không cố định thứ tự.)

In [ ]:
import copy
import hashlib
import json
import math
import os
import platform
import random
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Audio, display
from scipy.io import wavfile
from scipy.signal import resample_poly
from sklearn.metrics import roc_auc_score, roc_curve
from torch import nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = CONFIG["use_amp"] and DEVICE.type == "cuda"
SAMPLE_RATE = CONFIG["sample_rate"]
NUM_SAMPLES = round(SAMPLE_RATE * CONFIG["window_seconds"])  # 40 000 với 2,5 s ở 16 kHz
OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(tempfile.gettempdir()) / "gptwake_cache"  # không nằm trong output của Kaggle
CACHE_DIR.mkdir(parents=True, exist_ok=True)

torch.backends.cudnn.benchmark = True  # cho cuDNN tự chọn thuật toán tích chập nhanh nhất


def cpu_name():
    """Tên CPU, để ghi kèm số đo thời gian (số đo trên hai máy khác nhau không so được)."""
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


CPU_NAME = cpu_name()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print(f"PyTorch {torch.__version__} | thiết bị: {DEVICE}", end="")
if DEVICE.type == "cuda":
    print(f" ({torch.cuda.get_device_name(0)})", end="")
print(f" | mixed precision: {USE_AMP}")
print(f"CPU: {CPU_NAME}, {os.cpu_count()} luồng")
print(f"Mỗi cửa sổ: {CONFIG['window_seconds']} s × {SAMPLE_RATE} Hz = {NUM_SAMPLES} mẫu")
print(f"Kết quả sẽ lưu vào: {OUTPUT_DIR.resolve()}")

## 3. Tìm dữ liệu

### Dữ liệu được sắp xếp thế nào

Notebook đọc được ba kiểu sắp xếp và **tự nhận ra** kiểu nào đang có.

**Kiểu 1 — Parquet (dataset Kaggle chính thức của nhóm: `cuonghoangquoc/gptwake`).** Mỗi tập là một hoặc vài file
Parquet; mỗi dòng là một đoạn âm 2,5 s:

```
<data_root>/
├── validation.parquet          200 dòng
├── holdout.parquet             700 dòng
└── train-00000.parquet … train-00029.parquet    mỗi đoạn train xuất hiện đúng một lần
```

| Cột | Ý nghĩa |
|---|---|
| `sample_id` | tên gốc của đoạn âm, ví dụ `validation/positive/positive_hanhh_00521_quieter.wav` |
| `label` | 1 = có "Hey GPT", 0 = không |
| `category` | `positive`, `hard_negative` hoặc `ordinary_negative` |
| `sample_rate` | 16 000 |
| `audio_wav` | nguyên nội dung file WAV (PCM16), dạng byte |
| `H00`, `H25`, `H50`, `H70` | `True` nếu đoạn này thuộc phân phối train đó |

Bốn phân phối train dùng chung phần lớn file, nên mỗi đoạn chỉ lưu một lần và các cột `Hxx` cho biết nó thuộc phân phối
nào. Chọn `"train_splits": ["H25"]` thì notebook chỉ lấy các dòng có `H25 = True`.

**Kiểu 2 — thư mục WAV** (do `scripts/create_train.py` tạo trong repo): `validation/`, `holdout/`, `train/H25/`, mỗi thư mục
có `positive/` và `negative/`. Nhãn lấy theo tên thư mục.

**Kiểu 3 — danh sách CSV** (do `scripts/export_kaggle_dataset.py` tạo): như kiểu 2 nhưng train là `train_pool/` +
`train/H25.csv`.

### Tên file nói gì

Tên file (phần cuối của `sample_id`) trong validation/holdout cho biết **loại** và **nhóm**, để mục 11 phân tích lỗi
theo từng nhóm:

| Tên file | Loại | Nhóm |
|---|---|---|
| `positive_dat_00012_faster.wav` | positive | `dat` (người thật) |
| `positive_synthetic_gb_00003.wav` | positive | `synthetic_gb` (giọng TTS Anh-Anh) |
| `hard_negative_duc_00020_placement_2.wav` | hard_negative | `duc` |
| `ordinary_negative_vi_vlog17_source_...wav` | ordinary_negative | `vi_vlog17` (một vlog tiếng Việt) |

Hàm `describe_file` bên dưới tách tên theo quy tắc: bỏ tiền tố loại, rồi lấy các từ cho tới khi gặp một số
hoặc các từ `batch`, `source`, `augmented`, `processed`. Với dữ liệu Parquet, cột **loại** lấy thẳng từ `category`.

In [ ]:
import io

import pyarrow.dataset as pads


def is_split_dir(path):
    """Một thư mục là "tập dữ liệu" (kiểu 2) nếu có cả positive/ và negative/."""
    return (path / "positive").is_dir() and (path / "negative").is_dir()


def is_parquet_root(path):
    """Thư mục gốc kiểu 1 (Parquet) có validation.parquet và holdout.parquet."""
    return (path / "validation.parquet").is_file() and (path / "holdout.parquet").is_file()


def walk_dirs(base, max_depth=4):
    """Duyệt các thư mục con của base (tối đa max_depth cấp), bỏ qua thư mục ẩn và positive/negative."""
    stack = [(Path(base), 0)]
    while stack:
        path, depth = stack.pop()
        yield path
        if depth >= max_depth:
            continue
        try:
            children = [Path(e.path) for e in os.scandir(path) if e.is_dir()]
        except OSError:
            continue
        for child in sorted(children, reverse=True):
            if not child.name.startswith(".") and child.name not in ("positive", "negative"):
                stack.append((child, depth + 1))


def find_data_root(hint=None):
    bases = [Path(hint)] if hint else [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent]
    for base in bases:
        if not base.exists():
            continue
        for d in walk_dirs(base):
            if is_parquet_root(d) or (is_split_dir(d / "validation") and is_split_dir(d / "holdout")):
                return d
    raise FileNotFoundError(
        "Không tìm thấy dữ liệu (validation.parquet + holdout.parquet, hoặc thư mục validation/ + holdout/). "
        "Trên Kaggle: kiểm tra đã Add Input dataset chưa. Hoặc ghi đường dẫn vào CONFIG['data_root']."
    )


def split_source(root, split):
    """Nguồn dữ liệu của validation hoặc holdout."""
    if is_parquet_root(root):
        return {"kind": "parquet", "files": [root / f"{split}.parquet"], "column": None}
    return {"kind": "folder", "path": root / split}


def find_train_source(root, name):
    """Nguồn dữ liệu train của phân phối `name` (ví dụ "H25")."""
    if is_parquet_root(root):
        files = sorted(root.glob("train-*.parquet"))
        if files:
            return {"kind": "parquet", "files": files, "column": name}
    for candidate in (root / "train" / name, root / name):
        if is_split_dir(candidate):
            return {"kind": "folder", "path": candidate}
    for candidate in (root / "train" / f"{name}.csv", root / f"{name}.csv"):
        if candidate.is_file():
            return {"kind": "csv", "path": candidate}
    if Path("/kaggle/input").exists():  # train được upload thành một dataset riêng
        for d in walk_dirs("/kaggle/input"):
            if d.name == name and is_split_dir(d):
                return {"kind": "folder", "path": d}
            if (d / f"{name}.csv").is_file():
                return {"kind": "csv", "path": d / f"{name}.csv"}
    raise FileNotFoundError(f"Không thấy dữ liệu train {name} trong {root}.")


def describe_source(source):
    if source["kind"] == "parquet":
        where = f"lọc cột {source['column']}" if source["column"] else "toàn bộ"
        return f"{len(source['files'])} file Parquet ({source['files'][0].name}…), {where}"
    return f"{source['kind']}: {source['path']}"


CATEGORIES = ("hard_negative", "ordinary_negative", "positive")
STOP_WORDS = {"batch", "source", "augmented", "processed"}


def describe_file(name):
    """Tên file -> (loại, nhóm). Xem bảng ở ô markdown phía trên."""
    stem = Path(name).stem
    for category in CATEGORIES:
        if stem.startswith(category + "_"):
            words = []
            for word in stem[len(category) + 1 :].split("_"):
                if word.isdigit() or word in STOP_WORDS:
                    break
                words.append(word)
            return category, "_".join(words) or "khác"
    return "khác", "khác"


def parquet_dataset(source):
    dataset = pads.dataset([str(f) for f in source["files"]], format="parquet")
    row_filter = pads.field(source["column"]) if source["column"] else None  # cột Hxx kiểu bool
    return dataset, row_filter


def list_split(source):
    """Liệt kê các đoạn âm của một nguồn thành bảng: path, name, label, category, group."""
    if source["kind"] == "parquet":
        dataset, row_filter = parquet_dataset(source)
        meta = dataset.to_table(columns=["sample_id", "label", "category"], filter=row_filter).to_pandas()
        df = pd.DataFrame({"path": meta["sample_id"].astype(str), "label": meta["label"].astype(int)})
        df["name"] = [Path(p).name for p in df["path"]]
        df["category"] = meta["category"].astype(str).to_numpy()
        df["group"] = [describe_file(n)[1] for n in df["name"]]
    else:
        if source["kind"] == "csv":
            table = pd.read_csv(source["path"])
            paths = [(source["path"].parent / p).resolve() for p in table["path"]]
            labels = table["label"].astype(int).tolist()
        else:
            paths, labels = [], []
            for label, folder in ((1, "positive"), (0, "negative")):
                files = sorted((source["path"] / folder).glob("*.wav"))
                paths += files
                labels += [label] * len(files)
        df = pd.DataFrame({"path": [str(p) for p in paths], "label": labels})
        df["name"] = [Path(p).name for p in df["path"]]
        df[["category", "group"]] = [describe_file(n) for n in df["name"]]
        # Ở tập train, tên file không ghi người nói; loại suy ra từ nhãn cho positive.
        df.loc[(df["label"] == 1) & (df["category"] == "khác"), "category"] = "positive"
    if df.empty:
        raise FileNotFoundError(f"Không có đoạn âm nào trong {describe_source(source)}")
    if df["path"].duplicated().any():
        raise ValueError("Có đoạn âm bị trùng tên trong cùng một tập.")
    return df


def smoke_subset(df, per_class, seed=0):
    """Lấy ngẫu nhiên `per_class` đoạn mỗi nhãn (dùng khi smoke_test)."""
    parts = [g.sample(min(per_class, len(g)), random_state=seed) for _, g in df.groupby("label")]
    return pd.concat(parts).sort_index().reset_index(drop=True)


DATA_ROOT = find_data_root(CONFIG["data_root"])
VAL_SOURCE, HOLD_SOURCE = split_source(DATA_ROOT, "validation"), split_source(DATA_ROOT, "holdout")
val_df, hold_df = list_split(VAL_SOURCE), list_split(HOLD_SOURCE)
if CONFIG["smoke_test"]:
    val_df = smoke_subset(val_df, CONFIG["smoke_files_per_class"] // 2)
    hold_df = smoke_subset(hold_df, CONFIG["smoke_files_per_class"] // 2)

print("Thư mục dữ liệu:", DATA_ROOT)
for split in CONFIG["train_splits"]:
    print(f"Train {split}:", describe_source(find_train_source(DATA_ROOT, split)))
summary = pd.concat(
    {"validation": val_df.groupby("category").size(), "holdout": hold_df.groupby("category").size()},
    axis=1,
).fillna(0).astype(int)
display(summary)

## 4. Đọc âm thanh vào bộ nhớ

### File WAV chứa gì

Một file WAV mono 16 kHz, 16-bit là một dãy **số nguyên 16 bit** (từ −32 768 đến 32 767), 16 000 số cho mỗi giây.
Mỗi số là độ lệch của màng micro tại một thời điểm. Đoạn 2,5 s của nhóm có đúng 40 000 số (80 044 byte, gồm 44 byte
đầu file). Với dataset Parquet, cột `audio_wav` chứa chính các byte đó; notebook giải mã y như đọc file.

### Vì sao đọc hết vào RAM

Tập train có khoảng 46 000 đoạn. Nếu mỗi epoch lại giải mã 46 000 đoạn thì GPU phải chờ. Ta đọc **một lần**
vào một mảng `int16` kích thước (số_đoạn, 40 000): khoảng 3,7 GB, vừa với RAM của Kaggle (~30 GB).
Giữ dạng `int16` (2 byte/số) thay vì `float32` (4 byte/số) để tiết kiệm một nửa bộ nhớ. Khi đưa lên GPU mới đổi
sang số thực và chia cho 32 768 để về khoảng [−1, 1].

Với Parquet, notebook đọc theo từng khối 512 dòng, chỉ giải mã các dòng cần (ví dụ có `H25 = True`), và đặt mỗi đoạn
vào đúng hàng theo `sample_id`, nên thứ tự luôn khớp bảng nhãn. Đọc hết tập train mất khoảng một đến vài phút.

Hàm `read_wav` còn xử lý vài trường hợp lạ cho an toàn: WAV stereo (lấy trung bình hai kênh), WAV số thực,
WAV khác 16 kHz (đổi tần số lấy mẫu), đoạn dài/ngắn hơn 2,5 s (cắt hoặc đệm số 0 ở cuối).

Mảng đọc xong được lưu tạm vào thư mục cache. Lần chạy sau trong cùng phiên đọc lại rất nhanh.

In [ ]:
def read_wav(source):
    """Đọc một WAV (đường dẫn hoặc file trong bộ nhớ) -> (mảng int16 dài đúng NUM_SAMPLES, số mẫu gốc)."""
    rate, data = wavfile.read(source)
    if data.ndim == 2:  # stereo -> mono
        data = data.mean(axis=1)
    if np.issubdtype(data.dtype, np.floating):
        data = data * 32767.0
    elif data.dtype == np.int32:
        data = data / 65536.0
    elif data.dtype == np.uint8:
        data = (data.astype(np.float32) - 128.0) * 256.0
    if rate != SAMPLE_RATE:
        g = math.gcd(int(rate), SAMPLE_RATE)
        data = resample_poly(np.asarray(data, np.float64), SAMPLE_RATE // g, int(rate) // g)
    if data.dtype != np.int16:
        data = np.clip(np.round(data), -32768, 32767).astype(np.int16)
    out = np.zeros(NUM_SAMPLES, np.int16)
    n = min(len(data), NUM_SAMPLES)
    out[:n] = data[:n]
    return out, len(data)


def load_audio(df, tag, source):
    """Đọc mọi đoạn trong bảng df thành mảng int16 (số_đoạn, NUM_SAMPLES), có cache."""
    key = hashlib.md5(("|".join(df["path"]) + str(NUM_SAMPLES)).encode()).hexdigest()[:12]
    cache = CACHE_DIR / f"{tag}_{key}.npy"
    if cache.exists():
        return np.load(cache)
    X = np.zeros((len(df), NUM_SAMPLES), np.int16)
    lengths = np.full(len(df), -1, np.int64)
    start = time.time()
    if source["kind"] == "parquet":
        row_of = {sample_id: i for i, sample_id in enumerate(df["path"])}
        dataset, row_filter = parquet_dataset(source)
        done = 0
        for batch in dataset.to_batches(columns=["sample_id", "audio_wav"], filter=row_filter, batch_size=512):
            ids, blobs = batch.column(0).to_pylist(), batch.column(1)
            for j, sample_id in enumerate(ids):
                i = row_of.get(sample_id)
                if i is None:  # dòng không được chọn (ví dụ khi smoke_test)
                    continue
                X[i], lengths[i] = read_wav(io.BytesIO(blobs[j].as_py()))
                done += 1
                if done % 10_000 == 0:
                    print(f"  {tag}: đã đọc {done:,}/{len(df):,} đoạn ({time.time() - start:.0f} s)")
        if (lengths < 0).any():
            raise RuntimeError(f"{tag}: thiếu {int((lengths < 0).sum())} đoạn khi đọc Parquet.")
    else:
        with ThreadPoolExecutor(max_workers=16) as pool:
            for i, (wave, n) in enumerate(pool.map(read_wav, df["path"])):
                X[i], lengths[i] = wave, n
                if (i + 1) % 10_000 == 0:
                    print(f"  {tag}: đã đọc {i + 1:,}/{len(df):,} file ({time.time() - start:.0f} s)")
    wrong = int((lengths != NUM_SAMPLES).sum())
    print(
        f"{tag}: {len(df):,} đoạn, {X.nbytes / 1e9:.2f} GB, {time.time() - start:.0f} s. "
        f"Đoạn khác {NUM_SAMPLES} mẫu (đã cắt/đệm): {wrong}"
    )
    np.save(cache, X)
    return X


X_val, y_val = load_audio(val_df, "validation", VAL_SOURCE), val_df["label"].to_numpy()
X_hold, y_hold = load_audio(hold_df, "holdout", HOLD_SOURCE), hold_df["label"].to_numpy()

## 5. Xem và nghe thử dữ liệu

Luôn nghe vài mẫu trước khi train: nhãn có đúng không, âm có bị rè hay câm không.
Dưới đây là hai mẫu positive và hai mẫu negative của tập validation, kèm hình dạng sóng.
Trục ngang là thời gian (giây), trục dọc là biên độ trong khoảng [−1, 1].

In [ ]:
def show_examples(df, X, per_label=2, seed=0):
    rng = np.random.default_rng(seed)
    picks = [i for label in (1, 0) for i in rng.choice(np.flatnonzero(df["label"] == label), per_label, replace=False)]
    fig, axes = plt.subplots(len(picks), 1, figsize=(10, 1.6 * len(picks)), sharex=True)
    t = np.arange(NUM_SAMPLES) / SAMPLE_RATE
    for ax, i in zip(axes, picks):
        ax.plot(t, X[i] / 32768.0, lw=0.4)
        ax.set_ylim(-1, 1)
        ax.set_title(f"{df['name'].iloc[i]}  (nhãn {df['label'].iloc[i]})", fontsize=9, loc="left")
    axes[-1].set_xlabel("giây")
    plt.tight_layout()
    plt.show()
    for i in picks:
        print(df["name"].iloc[i])
        display(Audio(X[i] / 32768.0, rate=SAMPLE_RATE))


show_examples(val_df, X_val)

## 6. Mô hình XLS-R 300M

### 6.1 Nguồn gốc: học từ tiếng nói không cần nhãn

Ba mô hình trước (M5, SincConv, BC-ResNet) học **từ đầu** trên dữ liệu của nhóm. Mọi hiểu biết về giọng nói của chúng
đến từ khoảng 46 nghìn đoạn train, trong đó giọng người thật chỉ đến từ hai người. Kết quả cho thấy chúng nhận giọng TTS
gần như hoàn hảo nhưng bỏ sót nhiều giọng người lạ.

**wav2vec 2.0** (Baevski, Zhou, Mohamed, Auli, NeurIPS 2020, Meta AI) đề xuất cách học từ âm thanh **không có nhãn**:
mạng nghe một đoạn tiếng nói bị che mất vài khúc và phải đoán khúc bị che là gì, chọn đúng trong nhiều phương án.
Muốn đoán đúng, mạng phải hiểu âm vị, ngữ điệu, cách phát âm của nhiều người. Đây gọi là **học tự giám sát**
(self-supervised), giống cách mô hình ngôn ngữ học bằng cách đoán từ bị che.

**XLS-R** (Babu và cộng sự, 2021, Meta AI) là wav2vec 2.0 học trên khoảng **436 nghìn giờ** tiếng nói của **128 ngôn ngữ**
(gồm tiếng Việt và tiếng Anh), từ nhiều nguồn: phát biểu ở nghị viện châu Âu, sách nói, giọng đọc tình nguyện, điện thoại…
Có ba cỡ: 300 triệu, 1 tỉ và 2 tỉ tham số. Notebook dùng bản nhỏ nhất, **300M**.

**Vì sao đáng thử cho GPTWake:** XLS-R đã "nghe" hàng chục nghìn người nói khác nhau trước khi thấy dữ liệu của nhóm,
nên đặc trưng của nó ít phụ thuộc vào một giọng cụ thể. Đó đúng là điểm yếu của ba mô hình trước.

**Cách dùng phổ biến:** giữ nguyên phần lớn mạng, gắn thêm một đầu phân loại nhỏ và tinh chỉnh với ít dữ liệu có nhãn.
Bộ đánh giá SUPERB (Yang và cộng sự, Interspeech 2021) dùng cách này cho nhiều tác vụ tiếng nói, trong đó có keyword spotting.

### 6.2 Bên trong XLS-R

Đi từ waveform tới đầu ra, cho một cửa sổ 2,5 s:

| Bước | Thao tác | Kích thước đầu ra | Tham số |
|---|---|---|---|
| Chuẩn hoá | Trừ trung bình, chia độ lệch chuẩn (từng đoạn) | 40 000 mẫu | 0 |
| Bộ mã hoá CNN | 7 lớp tích chập 1D, 512 kênh; kernel 10, 3, 3, 3, 3, 2, 2; stride 5, 2, 2, 2, 2, 2, 2 | 512 × 124 khung | 4,2 triệu |
| Chiếu đặc trưng | 512 → 1 024 | 1 024 × 124 | 0,5 triệu |
| Mã hoá vị trí | Tích chập 1D rộng 128 khung (cho Transformer biết thứ tự) | 1 024 × 124 | 8,4 triệu |
| Transformer | **24 lớp**, mỗi lớp 16 đầu attention, tầng ẩn 4 096 | 1 024 × 124 | 302 triệu |
| **Tổng XLS-R** | | | **315 triệu** |

- **Bộ mã hoá CNN** làm việc giống lớp đầu của M5: đọc waveform, mỗi khung đầu ra nhìn **25 ms** âm thanh và cách nhau
  **20 ms**. Lớp này luôn bị đóng băng khi tinh chỉnh, như mọi công thức tinh chỉnh wav2vec 2.0.
- **Transformer:** mỗi khung "nhìn" được **mọi khung khác** trong cửa sổ qua cơ chế attention, nên ngữ cảnh là cả 2,5 s,
  không bị giới hạn như vùng tiếp nhận của mạng tích chập (M5 0,22 s, SincConv 0,57 s, BC-ResNet 1,2 s).

### 6.3 Đầu phân loại và cách tinh chỉnh

Lớp `XLSRWakeWordModel` của repo (file `src/gptwake/models/xlsr.py`) gắn lên XLS-R một đầu phân loại:

1. **Trộn các lớp** (`use_weighted_layer_sum`): lấy đầu ra của cả 25 tầng (đầu ra CNN đã chiếu + 24 lớp Transformer)
   rồi cộng có trọng số; 25 trọng số này cũng được học. Lý do: các lớp giữa của wav2vec 2.0 thường giữ thông tin âm vị
   tốt nhất, lớp trên cùng thiên về tác vụ học trước. Để mạng tự chọn thay vì đoán. Mục 12 vẽ các trọng số học được.
2. **Chiếu xuống** 256 chiều, **lấy trung bình** theo thời gian, rồi một lớp tuyến tính ra **2 điểm**
   (không phải / là "Hey GPT").
3. Pipeline chung của nhóm cần **một logit**. Lớp bọc `XLSRBinary` lấy hiệu hai điểm: logit = điểm "là" − điểm "không".
   Khi đó sigmoid(logit) đúng bằng xác suất softmax của lớp "Hey GPT", và loss BCE đúng bằng cross-entropy hai lớp,
   nên không thay đổi gì về toán học.

**Ba mức tinh chỉnh** (`finetune_mode`):

| Mức | Phần được train | Số tham số train | Khi nào dùng |
|---|---|---|---|
| `"probe"` | chỉ đầu phân loại | khoảng 0,26 triệu | nhanh nhất; đo xem đặc trưng học trước tốt tới đâu |
| `"partial"` (mặc định) | đầu + 4 lớp Transformer trên cùng | khoảng 50,6 triệu | cân bằng; ít nguy cơ học vẹt |
| `"full"` | mọi thứ trừ bộ mã hoá CNN | khoảng 311 triệu | mạnh nhất nhưng chậm, dễ học vẹt với dữ liệu nhỏ |

Để tiết kiệm dung lượng, `model.pt` **chỉ lưu phần được train** (khoảng 200 MB với `"partial"`). Khi nạp lại, notebook tải XLS-R
gốc rồi đè phần đã tinh chỉnh lên.

### 6.4 Cái giá: kích thước và tốc độ

| Mô hình | Tham số | Phép nhân-cộng / cửa sổ 2,5 s |
|---|---|---|
| BC-ResNet-3 | 53 nghìn | 39 triệu |
| SincConv | 119 nghìn | 59 triệu |
| M5 | 565 nghìn | 226 triệu |
| **XLS-R 300M** | **316 triệu** | **khoảng 44,7 tỉ** |

Thử trên một CPU máy chủ 2,1 GHz (không phải Kaggle), một cửa sổ mất khoảng **0,86 s với 1 luồng**. Khi chạy liên tục với bước
0,25 s, mỗi cửa sổ phải xử lý xong trong **dưới 250 ms**, nên XLS-R 300M nhiều khả năng **không chạy kịp thời gian thực trên laptop**
nếu không thu nhỏ (chưng cất, lượng tử hoá, cắt bớt lớp…). Vì vậy notebook mặc định chỉ bấm giờ CPU trên 100 clip
(`cpu_test_clips`); điểm số trên holdout vẫn tính cho đủ 700 clip (chấm bằng GPU).

Nói cách khác, XLS-R ở đây đóng vai **mốc trên**: nếu nó giảm mạnh tỉ lệ bỏ sót giọng người lạ thì đó là bằng chứng nút thắt
nằm ở dữ liệu/giọng, và hướng đi tiếp là đưa hiểu biết đó vào mô hình nhỏ (thêm giọng thật, hoặc chưng cất từ XLS-R sang BC-ResNet).

### 6.5 So với ba mô hình kia: điều gì không công bằng

| | M5 / SincConv / BC-ResNet | XLS-R |
|---|---|---|
| Dữ liệu train, validation, holdout | như nhau | như nhau |
| Chọn checkpoint, chọn ngưỡng, chỉ số | như nhau | như nhau |
| Dữ liệu **ngoài** | không | **436 nghìn giờ tiếng nói không nhãn** (qua bước học trước) |
| Tăng cường khi train | không / SpecAugment (BC-ResNet) | che khung + bỏ lớp (công thức chuẩn của wav2vec 2.0) |
| Kích thước | 53k–565k tham số | 316 triệu |

Vì có dữ liệu ngoài, XLS-R không phải một đối thủ "cùng điều kiện". Kết quả của nó trả lời câu hỏi khác: *nếu có hiểu biết về
rất nhiều giọng người thật thì làm được tới đâu?*

Ô dưới gồm **hai file nguyên văn** của repo: `audio.py` (chuẩn hoá waveform) và `xlsr.py` (mô hình). Dòng
`from .audio import …` của `xlsr.py` bị bỏ vì trong notebook hai hàm đó đã có sẵn ở ô trên.

In [ ]:
"""Waveform helpers shared by models that read raw 16 kHz audio."""

from __future__ import annotations

import torch


def sample_mask(waveforms: torch.Tensor, lengths: torch.Tensor | None) -> torch.Tensor:
    """Boolean mask of shape (batch, samples) that is True on real (unpadded) samples."""
    batch, samples = waveforms.shape
    if lengths is None:
        return torch.ones(batch, samples, dtype=torch.bool, device=waveforms.device)
    lengths = lengths.to(device=waveforms.device, dtype=torch.long)
    if (lengths <= 0).any() or (lengths > samples).any():
        raise ValueError("Each length must lie in [1, padded length].")
    return torch.arange(samples, device=waveforms.device)[None, :] < lengths[:, None]


def normalize_waveforms(
    waveforms: torch.Tensor, lengths: torch.Tensor | None = None, eps: float = 1e-7
) -> torch.Tensor:
    """Zero-mean, unit-variance normalization per utterance over its real samples.

    Statistics ignore right padding, and padded positions are set back to zero.
    """
    mask = sample_mask(waveforms, lengths).to(waveforms.dtype)
    count = mask.sum(dim=1, keepdim=True)
    mean = (waveforms * mask).sum(dim=1, keepdim=True) / count
    var = (((waveforms - mean) * mask) ** 2).sum(dim=1, keepdim=True) / count
    return (waveforms - mean) / torch.sqrt(var + eps) * mask

In [ ]:
"""XLS-R (wav2vec 2.0) wake-word classifier that reads raw 16 kHz waveforms.

The model maps a batch of mono waveforms in the amplitude-time domain directly to
class logits; no MFCC or log-mel frontend is computed. The pipeline is:

    waveform -> per-utterance normalization -> CNN feature encoder (20 ms frames)
    -> feature projection -> Transformer encoder -> (weighted) layer sum
    -> projector -> masked mean pooling over time -> linear classifier -> logits

Label 1 is the wake phrase ("Hey GPT"), label 0 is everything else. Window length,
inference hop and decision thresholds are deliberately not fixed here: they belong
to the data, evaluation and runtime layers and stay configurable.

Requires the optional ``xlsr`` dependency group (``uv sync --extra xlsr``). Model
weights are downloaded by ``transformers`` only when :meth:`from_pretrained` runs.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

import torch
from torch import nn
from transformers import Wav2Vec2Config, Wav2Vec2Model


SAMPLE_RATE = 16_000
DEFAULT_CHECKPOINT = "facebook/wav2vec2-xls-r-300m"
WAKE_LABEL = 1

FineTuneMode = Literal["probe", "partial", "full"]


@dataclass
class XLSRWakeWordConfig:
    """Hyperparameters of the classification head and the fine-tuning strategy.

    ``finetune_mode`` decides which backbone parameters receive gradients:

    - ``"probe"``: the whole backbone is frozen; only layer weights, projector and
      classifier train (a linear probe on self-supervised features).
    - ``"partial"``: the top ``trainable_top_layers`` Transformer layers (and the
      final encoder LayerNorm of stable-layer-norm models such as XLS-R) train.
    - ``"full"``: everything trains except the CNN feature encoder, which is the
      usual practice for wav2vec 2.0 fine-tuning.

    ``mask_time_prob`` and ``layerdrop`` override the checkpoint's SpecAugment-style
    time masking and LayerDrop rates; ``None`` keeps the checkpoint values.
    """

    checkpoint: str = DEFAULT_CHECKPOINT
    num_labels: int = 2
    use_weighted_layer_sum: bool = True
    projector_dim: int = 256
    classifier_dropout: float = 0.1
    finetune_mode: FineTuneMode = "partial"
    trainable_top_layers: int = 4
    mask_time_prob: float | None = None
    layerdrop: float | None = None
    normalize_waveform: bool = True
    gradient_checkpointing: bool = False

    def __post_init__(self):
        if self.finetune_mode not in ("probe", "partial", "full"):
            raise ValueError(f"Unknown finetune_mode: {self.finetune_mode!r}")
        if self.num_labels < 2:
            raise ValueError("num_labels must be at least 2.")
        if self.trainable_top_layers < 0:
            raise ValueError("trainable_top_layers must be non-negative.")


class XLSRWakeWordModel(nn.Module):
    """Binary (or multi-class) utterance classifier on top of a wav2vec 2.0 backbone."""

    def __init__(self, backbone: Wav2Vec2Model, config: XLSRWakeWordConfig | None = None):
        super().__init__()
        self.config = config or XLSRWakeWordConfig()
        self.backbone = backbone
        hidden = backbone.config.hidden_size
        states = backbone.config.num_hidden_layers + 1  # projected CNN features + each layer
        self.layer_weights = (
            nn.Parameter(torch.zeros(states)) if self.config.use_weighted_layer_sum else None
        )
        self.projector = nn.Linear(hidden, self.config.projector_dim)
        self.dropout = nn.Dropout(self.config.classifier_dropout)
        self.classifier = nn.Linear(self.config.projector_dim, self.config.num_labels)
        if self.config.gradient_checkpointing:
            self.backbone.gradient_checkpointing_enable()
        self.apply_finetune_mode()

    # Construction -----------------------------------------------------------------

    @classmethod
    def from_pretrained(
        cls, config: XLSRWakeWordConfig | None = None, **hf_kwargs
    ) -> XLSRWakeWordModel:
        """Load pretrained backbone weights (downloads on first use) and add a fresh head."""
        config = config or XLSRWakeWordConfig()
        overrides = {
            key: value
            for key, value in (
                ("mask_time_prob", config.mask_time_prob),
                ("layerdrop", config.layerdrop),
            )
            if value is not None
        }
        backbone = Wav2Vec2Model.from_pretrained(config.checkpoint, **overrides, **hf_kwargs)
        return cls(backbone, config)

    @classmethod
    def from_backbone_config(
        cls, backbone_config: Wav2Vec2Config, config: XLSRWakeWordConfig | None = None
    ) -> XLSRWakeWordModel:
        """Randomly initialized backbone, for tests and architecture experiments."""
        return cls(Wav2Vec2Model(backbone_config), config)

    # Fine-tuning ----------------------------------------------------------------

    def head_parameters(self) -> list[nn.Parameter]:
        params = list(self.projector.parameters()) + list(self.classifier.parameters())
        return ([self.layer_weights] if self.layer_weights is not None else []) + params

    def apply_finetune_mode(self) -> None:
        """Set ``requires_grad`` on the backbone according to ``config.finetune_mode``."""
        mode = self.config.finetune_mode
        backbone = self.backbone
        for param in backbone.parameters():
            param.requires_grad = mode == "full"
        if mode == "full":
            backbone.freeze_feature_encoder()
        elif mode == "partial":
            layers = backbone.encoder.layers
            top = min(self.config.trainable_top_layers, len(layers))
            for layer in layers[len(layers) - top :] if top else []:
                for param in layer.parameters():
                    param.requires_grad = True
            # Stable-layer-norm models (XLS-R) apply encoder.layer_norm after the last
            # layer, so it belongs to the trainable top; otherwise it sits below layer 0.
            if backbone.config.do_stable_layer_norm and top:
                for param in backbone.encoder.layer_norm.parameters():
                    param.requires_grad = True
        for param in self.head_parameters():
            param.requires_grad = True

    def parameter_groups(
        self, head_lr: float, backbone_lr: float, weight_decay: float = 0.0
    ) -> list[dict]:
        """Optimizer groups: a fresh head usually needs a larger LR than the backbone."""
        head = self.head_parameters()
        head_ids = {id(p) for p in head}
        backbone = [
            p for p in self.backbone.parameters() if p.requires_grad and id(p) not in head_ids
        ]
        groups = [{"params": head, "lr": head_lr, "weight_decay": weight_decay}]
        if backbone:
            groups.append({"params": backbone, "lr": backbone_lr, "weight_decay": weight_decay})
        return groups

    # Forward --------------------------------------------------------------------

    def frame_lengths(self, sample_lengths: torch.Tensor) -> torch.Tensor:
        """Number of 20 ms encoder frames produced for each input length in samples."""
        return self.backbone._get_feat_extract_output_lengths(sample_lengths).to(torch.long)

    def forward(self, waveforms: torch.Tensor, lengths: torch.Tensor | None = None) -> torch.Tensor:
        """Map waveforms of shape (batch, samples) at 16 kHz to logits (batch, num_labels).

        ``lengths`` gives the number of real samples per row when the batch is
        right-padded with zeros; omit it when every row is fully valid.
        """
        if waveforms.dim() != 2:
            raise ValueError("Expected mono waveforms shaped (batch, samples).")
        batch, samples = waveforms.shape
        if lengths is None:
            lengths = torch.full((batch,), samples, dtype=torch.long, device=waveforms.device)
        lengths = lengths.to(device=waveforms.device, dtype=torch.long)
        mask = sample_mask(waveforms, lengths)
        inputs = (
            normalize_waveforms(waveforms, lengths)
            if self.config.normalize_waveform
            else (waveforms * mask)
        )
        # Group-norm checkpoints were trained on zero padding without an attention mask.
        attention_mask = mask.long() if self.backbone.config.feat_extract_norm == "layer" else None
        outputs = self.backbone(
            inputs,
            attention_mask=attention_mask,
            output_hidden_states=self.layer_weights is not None,
            return_dict=True,
        )
        if self.layer_weights is not None:
            states = torch.stack(outputs.hidden_states, dim=0)  # (L+1, B, frames, hidden)
            weights = torch.softmax(self.layer_weights, dim=0).to(states.dtype)
            hidden = (weights[:, None, None, None] * states).sum(dim=0)
        else:
            hidden = outputs.last_hidden_state
        hidden = self.projector(hidden)
        frames = hidden.shape[1]
        valid = torch.arange(frames, device=hidden.device)[None, :] < self.frame_lengths(lengths)[
            :, None
        ].clamp(max=frames)
        valid = valid.to(hidden.dtype)
        pooled = (hidden * valid[..., None]).sum(dim=1) / valid.sum(dim=1, keepdim=True).clamp(
            min=1.0
        )
        return self.classifier(self.dropout(pooled))

    @torch.inference_mode()
    def wake_probability(
        self, waveforms: torch.Tensor, lengths: torch.Tensor | None = None
    ) -> torch.Tensor:
        """Softmax probability of the wake label for each row; call ``eval()`` first."""
        return torch.softmax(self(waveforms, lengths), dim=-1)[:, WAKE_LABEL]

### Vài hàm tiện ích để soi mô hình

Ba hàm dưới dùng ở ô tiếp theo và ở mục 13: in bảng kích thước từng lớp, tính vùng tiếp nhận
(mỗi vị trí đầu ra "nghe" được bao nhiêu mẫu âm thanh), và đếm số phép nhân-cộng cho một cửa sổ.

In [ ]:
def layer_table(model, layers):
    """Chạy thử một cửa sổ qua mô hình, in kích thước đầu ra và số tham số của từng lớp.

    layers: danh sách (tên hiển thị, module). Cột "kích thước đầu ra" là kênh × thời gian (mạng 1D) hoặc
    kênh × tần số × thời gian (mạng 2D). Cột "≈ bước" = độ dài cửa sổ ÷ số vị trí theo thời gian,
    gần đúng khoảng thời gian giữa hai vị trí liên tiếp.
    """
    shapes, hooks = {}, []
    for label, module in layers:
        hooks.append(module.register_forward_hook(lambda m, i, o, label=label: shapes.__setitem__(label, tuple(o.shape))))
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, NUM_SAMPLES))
    model.train(was_training)
    for h in hooks:
        h.remove()
    rows = [{"lớp": "đầu vào (waveform)", "kích thước đầu ra": f"1 × {NUM_SAMPLES}", "≈ bước": f"{1000 / SAMPLE_RATE:.4g} ms", "tham số": 0}]
    for label, module in layers:
        shape = shapes[label][1:]  # (kênh, thời gian) hoặc (kênh, tần số, thời gian)
        frames = shape[-1] if len(shape) in (2, 3) else None
        rows.append({
            "lớp": label,
            "kích thước đầu ra": " × ".join(map(str, shape)),
            "≈ bước": f"{1000 * CONFIG['window_seconds'] / frames:.3g} ms" if frames else "",
            "tham số": sum(p.numel() for p in module.parameters()),
        })
    table = pd.DataFrame(rows)
    display(table)
    print(f"Tổng tham số học được: {sum(p.numel() for p in model.parameters()):,}")


def receptive_field(geometry):
    """Vùng tiếp nhận của một vị trí đầu ra, từ danh sách (kernel, stride) theo thứ tự các lớp.

    Mỗi lớp có kernel k nhìn thêm (k - 1) bước của lớp trước; bước tích luỹ nhân với stride.
    """
    field, jump = 1, 1
    for kernel, stride in geometry:
        field += (kernel - 1) * jump
        jump *= stride
    return field, jump


def count_macs(model):
    """Số phép nhân-cộng (MAC) cho một cửa sổ: đo độ nặng tính toán, không phụ thuộc phần cứng."""
    total = 0

    def conv_hook(m, i, o):
        nonlocal total
        total += o.numel() * (m.in_channels // m.groups) * math.prod(m.kernel_size)

    def linear_hook(m, i, o):
        nonlocal total
        total += o.numel() * m.in_features

    def sinc_hook(m, i, o):
        nonlocal total
        total += o.numel() * m.kernel_size

    def logmel_hook(m, i, o):  # chỉ tính phép chiếu lên 40 dải mel, bỏ qua FFT
        nonlocal total
        total += o.numel() * m.mel_fb.shape[0]

    hooks = []
    for m in model.modules():
        if isinstance(m, (nn.Conv1d, nn.Conv2d)):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))
        elif type(m).__name__ == "SincConv1d":
            hooks.append(m.register_forward_hook(sinc_hook))
        elif type(m).__name__ == "LogMel":
            hooks.append(m.register_forward_hook(logmel_hook))
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, NUM_SAMPLES, device=next(model.parameters()).device))
    model.train(was_training)
    for h in hooks:
        h.remove()
    return total

### 6.6 Tải XLS-R và dựng mô hình

**Trên Kaggle cần bật Internet** để tải XLS-R (khoảng 1,3 GB, chỉ tải một lần mỗi phiên):
*Settings → Internet → On*. Kaggle chỉ cho bật Internet khi tài khoản đã **xác minh số điện thoại**
(*Settings* của tài khoản → *Phone verification*).

Không bật được Internet thì thêm XLS-R như một input: tìm trên Kaggle (*Add Input → Models* hoặc *Datasets*, từ khoá
`wav2vec2-xls-r-300m`) một bản có các file `config.json` và `pytorch_model.bin` hoặc `model.safetensors`. Notebook tự tìm
thư mục đó trong `/kaggle/input` nếu không tải được, hoặc ghi thẳng đường dẫn vào `MODEL_OPTIONS["checkpoint"]`.

Ô dưới tải XLS-R, in bảng tham số từng phần (tổng / được train) và số phép nhân-cộng.

In [ ]:
import transformers
from transformers import Wav2Vec2Model

transformers.logging.set_verbosity_error()  # bớt cảnh báo khi gắn đầu phân loại mới


def local_checkpoints():
    """Các thư mục trong /kaggle/input trông như một checkpoint wav2vec 2.0 (config.json + trọng số)."""
    found = []
    if Path("/kaggle/input").exists():
        for d in walk_dirs("/kaggle/input", max_depth=6):
            config = d / "config.json"
            weights = (d / "pytorch_model.bin").exists() or (d / "model.safetensors").exists()
            if config.is_file() and weights and "wav2vec2" in config.read_text(errors="ignore"):
                found.append(d)
    return found


def build_xlsr(config):
    """XLS-R + đầu phân loại. Thử checkpoint trong cấu hình, rồi tới bản tìm thấy trong /kaggle/input."""
    candidates = [config.checkpoint] + [str(d) for d in local_checkpoints() if str(d) != config.checkpoint]
    errors = []
    for name in candidates:
        try:
            inner = XLSRWakeWordModel.from_pretrained(XLSRWakeWordConfig(**{**asdict(config), "checkpoint": name}))
            return XLSRBinary(inner)
        except OSError as error:  # không có mạng, sai tên, thiếu file
            errors.append(f"{name}: {str(error).splitlines()[0]}")
    raise RuntimeError(
        "Không tải được XLS-R. Trên Kaggle: Settings → Internet → On (cần xác minh số điện thoại), "
        "hoặc Add Input một bản wav2vec2-xls-r-300m rồi ghi đường dẫn vào MODEL_OPTIONS['checkpoint'].\n"
        + "\n".join(errors)
    )


class XLSRBinary(nn.Module):
    """Bọc XLSRWakeWordModel (2 điểm) thành 1 logit: logit = điểm "Hey GPT" − điểm "không".

    sigmoid(logit) = softmax(điểm)["Hey GPT"], nên BCE trên logit này đúng bằng cross-entropy hai lớp.
    """

    def __init__(self, inner):
        super().__init__()
        self.inner = inner
        self.config = inner.config

    def forward(self, waveforms):
        z = self.inner(waveforms)
        return z[:, WAKE_LABEL] - z[:, 1 - WAKE_LABEL]

    def parameter_groups(self, head_lr, backbone_lr, weight_decay=0.0):
        return self.inner.parameter_groups(head_lr, backbone_lr, weight_decay)

    def checkpoint_state(self):
        """Chỉ phần được train; phần đóng băng giống hệt XLS-R gốc nên không cần lưu."""
        trainable = {name for name, p in self.named_parameters() if p.requires_grad}
        return {k: v for k, v in self.state_dict().items() if k in trainable}

    @torch.no_grad()
    def frame_scores(self, waveforms):
        """Logit của từng khung 20 ms. Đầu phân loại tuyến tính sau phép lấy trung bình, nên
        logit cả cửa sổ = trung bình các logit khung (khi eval, dropout tắt)."""
        m = self.inner
        x = normalize_waveforms(waveforms) if m.config.normalize_waveform else waveforms
        mask = torch.ones_like(x, dtype=torch.long) if m.backbone.config.feat_extract_norm == "layer" else None
        out = m.backbone(x, attention_mask=mask, output_hidden_states=m.layer_weights is not None, return_dict=True)
        if m.layer_weights is not None:
            states = torch.stack(out.hidden_states, dim=0)
            hidden = (torch.softmax(m.layer_weights, 0).to(states.dtype)[:, None, None, None] * states).sum(0)
        else:
            hidden = out.last_hidden_state
        z = m.classifier(m.projector(hidden))  # (B, khung, 2)
        return z[..., WAKE_LABEL] - z[..., 1 - WAKE_LABEL]


MODEL_CONFIG = XLSRWakeWordConfig(num_labels=2, **MODEL_OPTIONS)


def build_model():
    return build_xlsr(MODEL_CONFIG)


def load_model_from_checkpoint(ckpt):
    """Tải XLS-R gốc theo cấu hình lưu trong file, rồi đè phần đã tinh chỉnh lên."""
    model = build_xlsr(XLSRWakeWordConfig(**ckpt["model_config"]))
    missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)
    if unexpected:
        raise ValueError(f"model.pt có trọng số lạ: {unexpected[:5]}")
    return model


start = time.time()
demo = build_model()
print(f"Đã tải {MODEL_CONFIG.checkpoint} trong {time.time() - start:.0f} s")
parts = {
    "bộ mã hoá CNN": "inner.backbone.feature_extractor.",
    "chiếu đặc trưng": "inner.backbone.feature_projection.",
    "mã hoá vị trí": "inner.backbone.encoder.pos_conv_embed.",
    "các lớp Transformer": "inner.backbone.encoder.layers.",
    "LayerNorm cuối": "inner.backbone.encoder.layer_norm.",
    "đầu phân loại": "inner.",  # phần còn lại: layer_weights, projector, classifier
}
rows, seen = [], set()
for label, prefix in parts.items():
    params = [(n, p) for n, p in demo.named_parameters() if n.startswith(prefix) and n not in seen]
    seen.update(n for n, _ in params)
    rows.append({"phần": label, "tham số": sum(p.numel() for _, p in params),
                 "được train": sum(p.numel() for _, p in params if p.requires_grad)})
display(pd.DataFrame(rows))
total = sum(p.numel() for p in demo.parameters())
trainable = sum(p.numel() for p in demo.parameters() if p.requires_grad)
frames = int(demo.inner.frame_lengths(torch.tensor([NUM_SAMPLES])).item())
print(f"Tổng {total:,} tham số; được train {trainable:,} ({100 * trainable / total:.1f}%), chế độ {MODEL_CONFIG.finetune_mode!r}")
print(f"Số lớp Transformer: {demo.inner.backbone.config.num_hidden_layers}; mỗi cửa sổ {frames} khung 20 ms")
print(f"Phép nhân-cộng mỗi cửa sổ: {count_macs(demo) / 1e9:.1f} tỉ (chưa tính phép nhân trong attention)")
del demo
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 7. Vòng train

### Một bước học diễn ra thế nào

1. **Lấy một batch:** chọn ngẫu nhiên `batch_size` đoạn âm (mặc định 128) cùng nhãn của chúng.
2. **Lan truyền xuôi:** đưa batch qua mô hình, được 128 logit.
3. **Tính loss:** đo mức "sai" bằng *binary cross-entropy* (BCE). Với xác suất dự đoán $p$ và nhãn $y \in \{0, 1\}$:
   $$\text{BCE} = -\big[\,y \log p + (1-y)\log(1-p)\,\big]$$
   Bằng lời: nếu đoạn âm *có* "Hey GPT" ($y=1$) thì loss là $-\log p$, càng tự tin đúng ($p \to 1$) loss càng gần 0,
   tự tin sai ($p \to 0$) thì loss rất lớn. Tương tự với $y = 0$. Hàm `binary_cross_entropy_with_logits` gộp sigmoid
   và BCE thành một phép tính ổn định số học.
4. **Lan truyền ngược:** PyTorch tính đạo hàm của loss theo từng trọng số (cho biết chỉnh trọng số theo hướng nào để loss giảm).
5. **Cập nhật:** dùng **AdamW** (Adam có weight decay tách riêng), cách thường dùng khi tinh chỉnh Transformer,
   với **hai tốc độ học**:
   - Đầu phân loại (mới tinh, khởi tạo ngẫu nhiên): `learning_rate` = 0,001.
   - Các lớp XLS-R được tinh chỉnh: `backbone_lr` = 0,00005, nhỏ hơn 20 lần. Các lớp này đã học được nhiều điều có ích;
     bước lớn sẽ xoá mất hiểu biết đó (hiện tượng "quên thảm hoạ").

   Tốc độ học tăng dần trong nửa epoch đầu (warmup) rồi giảm theo đường cosin về 0, cập nhật sau mỗi bước.
   Mô hình học trước hội tụ nhanh, nên mặc định chỉ **4 epoch** và không dừng sớm.

Duyệt hết tập train một lượt gọi là **một epoch**. Mỗi epoch xáo trộn thứ tự để các batch khác nhau.

### Chọn epoch tốt nhất

Sau mỗi epoch, notebook chấm mô hình trên **validation** (không học gì từ tập này) và in:
loss, độ chính xác ở ngưỡng 0,5 và AUC (mục 8). Train lâu quá thì mô hình bắt đầu **học vẹt**: loss train vẫn
giảm nhưng loss validation tăng lên. Notebook nhớ lại trọng số ở epoch có **loss validation thấp nhất**
(đánh dấu `*` khi in) và dùng bản đó cho mọi bước sau. Nếu `patience` epoch liền không tốt hơn thì dừng sớm.

Validation chỉ có 200 file nên các con số dao động khá mạnh giữa các epoch. Đó là lý do chọn theo *loss*
(thay đổi mượt) chứ không theo độ chính xác (nhảy theo bậc 0,5%).

### Vài chi tiết kỹ thuật

- Dữ liệu nằm trong RAM dạng `int16`. Mỗi batch được chép lên GPU rồi mới đổi sang số thực, để giảm lượng dữ liệu chuyển.
- **Mixed precision** (`use_amp`): phần lớn phép tính chạy bằng số thực 16-bit cho nhanh gấp đôi; `GradScaler` nhân loss
  lên trước khi lan truyền ngược để đạo hàm nhỏ không bị làm tròn về 0.
- Không thêm tăng cường dữ liệu: dữ liệu của nhóm đã augment sẵn (đổi vị trí, nhỏ tiếng, thêm nhiễu, nói nhanh).
- Riêng XLS-R, khi train, chính backbone tự **che ngẫu nhiên một số đoạn khung** (`mask_time_prob`, kiểu SpecAugment trên đặc
  trưng) và **bỏ ngẫu nhiên cả lớp Transformer** (`layerdrop`), theo giá trị lưu trong checkpoint. Đây là công thức tinh chỉnh
  chuẩn của wav2vec 2.0; giống SpecAugment của BC-ResNet, nó là một phép chống học vẹt mà M5 và SincConv không có.

In [ ]:
def batches(X, y, batch_size, shuffle, rng=None):
    """Sinh các batch (waveform float trên DEVICE, nhãn float) từ mảng int16 trong RAM."""
    order = rng.permutation(len(X)) if shuffle else np.arange(len(X))
    for start in range(0, len(order), batch_size):
        idx = np.sort(order[start : start + batch_size])
        xb = torch.from_numpy(X[idx]).to(DEVICE, non_blocking=True).float().div_(32768.0)
        yb = torch.from_numpy(np.asarray(y[idx], np.float32)).to(DEVICE)
        yield xb, yb


def train_one_epoch(model, X, y, optimizer, scaler, rng, step_scheduler=None):
    model.train()
    total_loss, correct = 0.0, 0
    for xb, yb in batches(X, y, CONFIG["batch_size"], shuffle=True, rng=rng):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(xb)
        loss = F.binary_cross_entropy_with_logits(logits.float(), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if step_scheduler is not None:  # lịch lr đổi sau mỗi bước (warmup + cosine)
            step_scheduler.step()
        total_loss += loss.item() * len(yb)
        correct += ((logits.detach() > 0) == (yb > 0.5)).sum().item()
    return total_loss / len(X), correct / len(X)


@torch.no_grad()
def predict_logits(model, X, batch_size=None):
    """Logit của mọi đoạn trong X (mô hình ở chế độ eval: tắt dropout, BatchNorm dùng thống kê đã học)."""
    model.eval()
    out = []
    batch_size = batch_size or CONFIG.get("eval_batch_size", 256)
    for xb, _ in batches(X, np.zeros(len(X)), batch_size, shuffle=False):
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            out.append(model(xb).float().cpu())
    return torch.cat(out).numpy().astype(np.float64)

## 8. Cách đánh giá: ngưỡng, FRR, FAR, ROC

### Từ xác suất tới quyết định

Mô hình cho mỗi đoạn âm một **điểm** $s = \text{sigmoid}(\text{logit}) \in [0, 1]$. Chọn một **ngưỡng** $\theta$:
$s \ge \theta$ thì báo "có Hey GPT". So với nhãn thật được bốn ô (ma trận nhầm lẫn):

| | Mô hình nói **có** | Mô hình nói **không** |
|---|---|---|
| **Thật sự có** (positive) | TP: đúng | FN: **bỏ sót** |
| **Thật sự không** (negative) | FP: **báo nhầm** | TN: đúng |

Từ đó:
- **FRR** (false reject rate, tỉ lệ bỏ sót) $= \dfrac{FN}{TP + FN}$: trong các lần người dùng gọi, bao nhiêu phần trăm bị lờ đi.
- **FAR** (false accept rate, tỉ lệ báo nhầm) $= \dfrac{FP}{FP + TN}$: trong các đoạn không có "Hey GPT", bao nhiêu phần trăm bị bật nhầm.
- **Accuracy** $= \dfrac{TP + TN}{\text{tổng}}$. Dễ hiểu nhưng che mất loại lỗi nào đang xảy ra, nên luôn xem kèm FRR và FAR.
- **Precision** $= \dfrac{TP}{TP + FP}$, **recall** $= 1 - \text{FRR}$, **F1** là trung bình điều hoà của hai cái.

### Đường ROC và AUC

Quét ngưỡng từ 1 xuống 0 thì FAR tăng dần, FRR giảm dần. Vẽ mọi cặp (FAR, 1 − FRR) được **đường ROC**.
**AUC** là diện tích dưới đường này: xác suất để một đoạn positive ngẫu nhiên được chấm điểm cao hơn một
đoạn negative ngẫu nhiên. AUC = 1 là tách hoàn hảo, 0,5 là đoán mò. AUC không phụ thuộc ngưỡng.

**Đường DET** vẽ thẳng FRR theo FAR (thang log), hay dùng trong wake word vì nhìn rõ vùng FAR nhỏ.

### Chọn ngưỡng (trên validation, không bao giờ trên holdout)

- `"youden"`: chọn ngưỡng làm $(1 - \text{FRR}) - \text{FAR}$ lớn nhất, tức cân bằng hai loại lỗi.
- `"target_far"`: chọn ngưỡng có FRR nhỏ nhất trong các ngưỡng giữ FAR ≤ `target_far`.
  Sản phẩm thật thường làm kiểu này vì báo nhầm phiền hơn bỏ sót. Lưu ý validation chỉ có 100 negative,
  nên FAR trên validation chỉ đo được theo bậc 1%.
- `"fixed"`: dùng luôn `fixed_threshold`.

### FRR tại FAR cố định

Để so hai mô hình **công bằng**, hướng nghiên cứu của nhóm dùng "FRR khi FAR = x%": với mỗi mô hình, tìm trên
holdout ngưỡng sao cho FAR không vượt x%, rồi đọc FRR. Con số này đo khả năng *phân tách* của mô hình,
không phụ thuộc cách chọn ngưỡng. Ngược lại, các số ở ngưỡng đã chọn trên validation mô phỏng đúng
những gì người dùng sẽ gặp.

**Lưu ý về FA/h:** FAR ở đây tính trên từng clip 2,5 s rời rạc. Số lần báo nhầm **mỗi giờ** (FA/h) khi chạy
liên tục phải đo trên bản ghi dài với cửa sổ trượt (mục 14).

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def bce_from_logits(labels, logits):
    """Binary cross-entropy trung bình, tính ổn định: log(1 + e^z) - y z."""
    return float(np.mean(np.logaddexp(0.0, logits) - labels * logits))


def binary_report(labels, scores, threshold):
    labels = np.asarray(labels).astype(bool)
    pred = np.asarray(scores) >= threshold
    tp, fn = int((pred & labels).sum()), int((~pred & labels).sum())
    fp, tn = int((pred & ~labels).sum()), int((~pred & ~labels).sum())
    safe = lambda a, b: a / b if b else float("nan")
    precision, recall = safe(tp, tp + fp), safe(tp, tp + fn)
    return {
        "threshold": float(threshold),
        "accuracy": safe(tp + tn, tp + tn + fp + fn),
        "FRR": safe(fn, tp + fn),
        "FAR": safe(fp, fp + tn),
        "precision": precision,
        "recall": recall,
        "F1": safe(2 * precision * recall, precision + recall),
        "TP": tp, "FN": fn, "FP": fp, "TN": tn,
    }


def choose_threshold(labels, scores):
    mode = CONFIG["threshold_mode"]
    if mode == "fixed":
        return float(CONFIG["fixed_threshold"])
    fpr, tpr, thresholds = roc_curve(labels, scores)
    thresholds = np.minimum(thresholds, 1.0)  # phần tử đầu của sklearn là +inf
    if mode == "youden":
        best = int(np.argmax(tpr - fpr))
    elif mode == "target_far":
        allowed = np.flatnonzero(fpr <= CONFIG["target_far"])
        best = int(allowed[np.argmax(tpr[allowed])])
    else:
        raise ValueError(f"threshold_mode không hợp lệ: {mode}")
    return float(thresholds[best])


def frr_at_far(labels, scores, far):
    """FRR nhỏ nhất đạt được khi FAR không vượt `far` (đọc từ đường ROC)."""
    fpr, tpr, _ = roc_curve(labels, scores)
    return float(1.0 - tpr[fpr <= far].max())


def group_report(scored, threshold):
    """Tỉ lệ lỗi theo từng nhóm: FRR cho nhóm positive, FAR cho nhóm negative."""
    rows = []
    for (category, group), g in scored.groupby(["category", "group"]):
        wrong = (g["score"] >= threshold) != (g["label"] == 1)
        rows.append({
            "loại": category,
            "nhóm": group,
            "số file": len(g),
            "lỗi": int(wrong.sum()),
            "tỉ lệ lỗi": wrong.mean(),
            "nghĩa là": "FRR (bỏ sót)" if category == "positive" else "FAR (báo nhầm)",
        })
    return pd.DataFrame(rows)

## 9. Chạy thí nghiệm

Hàm `run_experiment(split, seed)` làm trọn một lần train:

1. Đọc dữ liệu train của phân phối `split` (H00/H25/H50/H70) vào RAM.
2. Dựng mô hình mới, đặt seed, train với chọn epoch theo loss validation.
3. Nạp lại trọng số tốt nhất, **chọn ngưỡng trên validation**.
4. **Test trên holdout bằng CPU, batch 1** (xem dưới), lưu mọi thứ vào `outputs/<mô hình>_<split>_seed<seed>/`:
   - `model.pt`: trọng số + cấu hình + ngưỡng (đủ để nạp lại và dùng ngay),
   - `history.csv`: loss/độ chính xác từng epoch,
   - `holdout_scores.csv`: điểm **và thời gian suy luận** của từng file holdout,
   - `metrics.json`: các chỉ số cuối cùng, gồm cả thống kê thời gian.

### Vì sao test trên CPU, batch 1?

Khi chạy thật trên laptop, hệ thống đưa **từng cửa sổ một** vào mô hình, thường **không có GPU**. Train dùng GPU và batch
lớn cho nhanh, nhưng để biết mô hình chạy thật nhanh tới đâu, phần test làm đúng như lúc dùng thật: mô hình được chép
sang CPU, mỗi clip holdout chạy riêng (batch = 1), và đồng hồ bấm **chỉ quanh lần gọi mô hình** (không tính đọc file).
Trước khi bấm giờ có `cpu_warmup` lần chạy làm nóng (lần đầu luôn chậm vì cấp phát bộ nhớ, nạp thư viện).

Test chạy một lần cho mỗi số luồng trong `cpu_threads` (mặc định 1 luồng và mọi luồng):
- **1 luồng** gần với thiết bị yếu hoặc khi máy đang bận việc khác; đây là con số nên đặt lên trục so sánh.
- **Mọi luồng** cho biết mô hình tận dụng CPU nhiều nhân tốt tới đâu.

Kết quả là cùng một lần test cho cả **độ chính xác** lẫn **thời gian**, để xem đánh đổi ở mục 13. Notebook cũng chấm
holdout trên GPU và kiểm tra hai bên cho cùng điểm (lệch rất nhỏ là bình thường vì GPU tính bằng số 16 bit).

### Chỉ test một checkpoint có sẵn (ví dụ để đo lại trên laptop)

Ghi đường dẫn vào `CONFIG["checkpoints_to_test"]`, ví dụ `["outputs/sincconv_H25_seed0/model.pt"]`, rồi *Run All*:
notebook **bỏ qua train**, nạp checkpoint, dùng ngưỡng đã lưu trong đó và chạy đúng phần test CPU này. Kết quả lưu vào
`outputs/<tên checkpoint>_test_<tên máy>/`. CPU Kaggle khác laptop, nên số cho báo cáo nên đo theo cách này trên máy của nhóm.

Vòng lặp cuối ô chạy mọi tổ hợp `train_splits × seeds` trong cấu hình. Thời gian mỗi epoch được in ra;
với GPU T4 và đủ 46 400 file, xem dòng in đầu tiên để ước lượng tổng thời gian.

In [ ]:
_TRAIN_CACHE = {}


def load_train(split):
    if split not in _TRAIN_CACHE:
        _TRAIN_CACHE.clear()  # chỉ giữ một phân phối trong RAM
        source = find_train_source(DATA_ROOT, split)
        df = list_split(source)
        if CONFIG["smoke_test"]:
            df = smoke_subset(df, CONFIG["smoke_files_per_class"])
        X = load_audio(df, f"train_{split}", source)
        _TRAIN_CACHE[split] = (df, X, df["label"].to_numpy())
    return _TRAIN_CACHE[split]


def cpu_batch1_test(model, X, threads, warmup):
    """Chạy từng clip một trên CPU với `threads` luồng. Trả (logit, thời gian ms) của mỗi clip."""
    cpu_model = copy.deepcopy(model).float().cpu().eval()
    original = torch.get_num_threads()
    torch.set_num_threads(threads)
    logits, times = np.zeros(len(X)), np.zeros(len(X))
    try:
        with torch.inference_mode():
            for i in range(warmup):
                cpu_model(torch.from_numpy(X[i % len(X)][None]).float().div_(32768.0))
            for i in range(len(X)):
                x = torch.from_numpy(X[i][None]).float().div_(32768.0)  # chuẩn bị đầu vào ngoài đồng hồ
                start = time.perf_counter()
                out = cpu_model(x)
                times[i] = (time.perf_counter() - start) * 1000.0
                logits[i] = float(out.reshape(-1)[0])
    finally:
        torch.set_num_threads(original)
    return logits, times


def cpu_thread_settings():
    return sorted({int(t or os.cpu_count() or 1) for t in CONFIG["cpu_threads"]})


def evaluate_on_holdout(model, threshold, meta):
    """Chấm holdout (CPU batch 1 nếu cpu_test), trả (dòng kết quả, bảng điểm từng file)."""
    fast_scores = sigmoid(predict_logits(model, X_hold))  # trên DEVICE, batch lớn
    latencies = {}
    # Mô hình lớn (XLS-R) chạy CPU rất lâu: chỉ bấm giờ trên `cpu_test_clips` clip đầu, điểm lấy từ lần chấm trên DEVICE.
    limit = CONFIG.get("cpu_test_clips")
    subset = np.arange(len(X_hold)) if not limit or limit >= len(X_hold) else np.arange(limit)
    if CONFIG["cpu_test"]:
        for threads in cpu_thread_settings():
            logits, ms = cpu_batch1_test(model, X_hold[subset], threads, CONFIG["cpu_warmup"])
            latencies[threads] = ms
            print(f"  test CPU batch 1, {threads} luồng, {len(subset)} clip: trung vị {np.median(ms):.2f} ms/clip, p90 {np.percentile(ms, 90):.2f} ms")
        cpu_scores = sigmoid(logits)
        diff = np.abs(cpu_scores - fast_scores[subset]).max()
        flips = int(((cpu_scores >= threshold) != (fast_scores[subset] >= threshold)).sum())
        print(f"  khớp với lần chấm trên {DEVICE.type}: lệch điểm lớn nhất {diff:.2e}, số quyết định khác nhau {flips}")
        if diff > 0.01:
            print("  CẢNH BÁO: điểm CPU và điểm trên thiết bị train lệch nhiều, kiểm tra lại mô hình.")
        hold_scores = cpu_scores if len(subset) == len(X_hold) else fast_scores
    else:
        hold_scores = fast_scores

    result = dict(meta)
    result.update(binary_report(y_hold, hold_scores, threshold))
    result["AUC"] = float(roc_auc_score(y_hold, hold_scores))
    for far in CONFIG["far_points"]:
        result[f"FRR@FAR={far:g}"] = frr_at_far(y_hold, hold_scores, far)
    result["params"] = sum(p.numel() for p in model.parameters())
    result["MMAC"] = count_macs(model) / 1e6
    result["cpu"] = CPU_NAME
    for threads, ms in latencies.items():
        result[f"latency_median_ms_{threads}t"] = float(np.median(ms))
        result[f"latency_mean_ms_{threads}t"] = float(np.mean(ms))
        result[f"latency_p90_ms_{threads}t"] = float(np.percentile(ms, 90))
        result[f"latency_p99_ms_{threads}t"] = float(np.percentile(ms, 99))
        result[f"rtf_{threads}t"] = float(np.median(ms) / 1000 / CONFIG["window_seconds"])

    scored = hold_df.drop(columns="path").assign(score=hold_scores, predicted=(hold_scores >= threshold).astype(int))
    for threads, ms in latencies.items():
        column = np.full(len(scored), np.nan)
        column[subset] = ms
        scored[f"latency_ms_{threads}t"] = column
    return result, scored


def save_run(run_dir, result, scored, history=None):
    run_dir.mkdir(parents=True, exist_ok=True)
    scored.to_csv(run_dir / "holdout_scores.csv", index=False)
    if history is not None:
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    (run_dir / "metrics.json").write_text(json.dumps(result, indent=2, ensure_ascii=False))


def make_optimizer(model, steps_per_epoch):
    """Bộ tối ưu và lịch lr theo CONFIG.

    Trả (optimizer, lịch đổi mỗi epoch hoặc None, lịch đổi mỗi bước hoặc None).
    - "adam" (M5, SincConv), "sgd" có momentum (BC-ResNet) hoặc "adamw" (XLS-R).
    - Có "backbone_lr" và mô hình có parameter_groups (XLS-R): đầu phân loại học với learning_rate,
      phần backbone học trước học với backbone_lr nhỏ hơn nhiều.
    - "step": nhân lr với lr_gamma sau mỗi lr_step_epochs epoch.
    - "cosine": tăng tuyến tính từ 0 trong warmup_epochs epoch đầu, rồi giảm theo nửa chu kỳ cosin về 0
      ở cuối epoch cuối cùng (như mã gốc của BC-ResNet, cập nhật sau mỗi bước).
    """
    lr, wd = CONFIG["learning_rate"], CONFIG["weight_decay"]
    if CONFIG.get("backbone_lr") is not None and hasattr(model, "parameter_groups"):
        params = model.parameter_groups(head_lr=lr, backbone_lr=CONFIG["backbone_lr"], weight_decay=wd)
    else:
        params = [p for p in model.parameters() if p.requires_grad]
    kind = CONFIG.get("optimizer", "adam")
    if kind == "sgd":
        optimizer = torch.optim.SGD(params, lr=lr, momentum=CONFIG.get("momentum", 0.9), weight_decay=wd)
    elif kind == "adamw":
        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    else:
        optimizer = torch.optim.Adam(params, lr=lr, weight_decay=wd)
    schedule = CONFIG.get("lr_schedule", "step" if CONFIG.get("lr_step_epochs") else None)
    if schedule == "cosine":
        warmup = max(1, round(CONFIG.get("warmup_epochs", 0) * steps_per_epoch))
        total = max(warmup + 1, CONFIG["epochs"] * steps_per_epoch)

        def factor(step):  # step = số bước đã đi
            if step < warmup:
                return (step + 1) / warmup
            return 0.5 * (1 + math.cos(math.pi * (step - warmup) / (total - warmup)))

        return optimizer, None, torch.optim.lr_scheduler.LambdaLR(optimizer, factor)
    if schedule == "step" and CONFIG.get("lr_step_epochs"):
        return optimizer, torch.optim.lr_scheduler.StepLR(optimizer, CONFIG["lr_step_epochs"], CONFIG["lr_gamma"]), None
    return optimizer, None, None


# GradScaler đôi khi bỏ một bước (đạo hàm tràn số) nên PyTorch cảnh báo lịch lr chạy trước optimizer: vô hại.
import warnings  # noqa: E402

warnings.filterwarnings("ignore", message="Detected call of `lr_scheduler.step\\(\\)` before `optimizer.step\\(\\)`")


def model_state(model):
    """Trọng số cần giữ: toàn bộ, hoặc chỉ phần được train nếu mô hình có checkpoint_state (XLS-R)."""
    state = model.checkpoint_state() if hasattr(model, "checkpoint_state") else model.state_dict()
    return {k: v.detach().clone() for k, v in state.items()}


def restore_state(model, state):
    model.load_state_dict(state, strict=not hasattr(model, "checkpoint_state"))


def run_experiment(split, seed):
    run_name = f"{MODEL_NAME}_{split}_seed{seed}"
    run_dir = OUTPUT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    train_df, X_train, y_train = load_train(split)
    print(f"\n===== {run_name}: {int(y_train.sum()):,} positive, {int((y_train == 0).sum()):,} negative =====")

    set_seed(seed)
    rng = np.random.default_rng(seed)
    model = build_model().to(DEVICE)
    steps_per_epoch = math.ceil(len(X_train) / CONFIG["batch_size"])
    optimizer, scheduler, step_scheduler = make_optimizer(model, steps_per_epoch)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    best_loss, best_epoch, best_state, wait, history = math.inf, 0, None, 0, []
    for epoch in range(1, CONFIG["epochs"] + 1):
        start = time.time()
        lr = optimizer.param_groups[0]["lr"]
        train_loss, train_acc = train_one_epoch(model, X_train, y_train, optimizer, scaler, rng, step_scheduler)
        if scheduler:
            scheduler.step()
        val_logits = predict_logits(model, X_val)
        val_scores = sigmoid(val_logits)
        row = {
            "epoch": epoch,
            "lr": lr,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": bce_from_logits(y_val, val_logits),
            "val_acc": float(((val_scores >= 0.5) == (y_val == 1)).mean()),
            "val_auc": float(roc_auc_score(y_val, val_scores)),
            "seconds": time.time() - start,
        }
        history.append(row)
        improved = row["val_loss"] < best_loss - 1e-4
        if improved:
            best_loss, best_epoch, wait = row["val_loss"], epoch, 0
            best_state = model_state(model)
        else:
            wait += 1
        print(
            f"epoch {epoch:3d} | lr {lr:.1e} | train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {row['val_loss']:.4f} acc {row['val_acc']:.3f} auc {row['val_auc']:.4f} | "
            f"{row['seconds']:.0f} s {'*' if improved else ''}"
        )
        if CONFIG["patience"] and wait >= CONFIG["patience"]:
            print(f"Dừng sớm: {wait} epoch liền loss validation không giảm.")
            break

    restore_state(model, best_state)
    val_scores = sigmoid(predict_logits(model, X_val))
    threshold = choose_threshold(y_val, val_scores)
    meta = {"run": run_name, "model": MODEL_NAME, "split": split, "seed": seed, "best_epoch": best_epoch}
    print(f"Chốt epoch {best_epoch}, ngưỡng {threshold:.3f}. Test trên holdout:")
    result, scored = evaluate_on_holdout(model, threshold, meta)
    val_report = binary_report(y_val, val_scores, threshold)
    result["val_FRR"], result["val_FAR"] = val_report["FRR"], val_report["FAR"]
    save_run(run_dir, result, scored, history)
    torch.save(
        {
            "state_dict": model_state(model),
            "model_name": MODEL_NAME,
            "model_config": asdict(model.config),
            "threshold": threshold,
            "config": CONFIG,
            "split": split,
            "seed": seed,
            "n_train": len(train_df),
        },
        run_dir / "model.pt",
    )
    print(
        f"Holdout: acc {result['accuracy']:.3f}, FRR {result['FRR']:.3f}, FAR {result['FAR']:.3f}, "
        f"AUC {result['AUC']:.4f}"
    )
    return result, model, pd.DataFrame(history), scored


def test_checkpoint(path):
    """Nạp model.pt đã train, chỉ chạy phần test (không train)."""
    path = Path(path)
    ckpt = torch.load(path, map_location="cpu")
    if ckpt["model_name"] != MODEL_NAME:
        raise ValueError(f"{path} là checkpoint của {ckpt['model_name']}; hãy test nó bằng notebook của mô hình đó.")
    model = load_model_from_checkpoint(ckpt).to(DEVICE)
    host = platform.node().split(".")[0] or "may"
    run_name = f"{path.parent.name}_test_{host}"
    print(f"\n===== Test {path} (ngưỡng {ckpt['threshold']:.3f}) =====")
    meta = {"run": run_name, "model": ckpt["model_name"], "split": ckpt.get("split"), "seed": ckpt.get("seed"), "best_epoch": None}
    result, scored = evaluate_on_holdout(model, ckpt["threshold"], meta)
    save_run(OUTPUT_DIR / run_name, result, scored)
    print(f"Holdout: acc {result['accuracy']:.3f}, FRR {result['FRR']:.3f}, FAR {result['FAR']:.3f}, AUC {result['AUC']:.4f}")
    return result, model, None, scored


RESULTS, RUNS = [], {}
jobs = (
    [(test_checkpoint, (p,)) for p in CONFIG["checkpoints_to_test"]]
    if CONFIG["checkpoints_to_test"]
    else [(run_experiment, (split, seed)) for split in CONFIG["train_splits"] for seed in CONFIG["seeds"]]
)
for job, args in jobs:
    result, model, history, scored = job(*args)
    RESULTS.append(result)
    RUNS[result["run"]] = {"model": model, "history": history, "scored": scored, "threshold": result["threshold"]}

LAST_RUN = RESULTS[-1]["run"]
model = RUNS[LAST_RUN]["model"]
threshold = RUNS[LAST_RUN]["threshold"]
# Chế độ chỉ test ghi ra file tóm tắt riêng, không đè kết quả của lần train.
SUMMARY_FILE = OUTPUT_DIR / (
    f"{MODEL_NAME}_test_{platform.node().split('.')[0] or 'may'}_summary.csv"
    if CONFIG["checkpoints_to_test"]
    else f"{MODEL_NAME}_summary.csv"
)

## 10. Đường cong học

Đọc biểu đồ thế nào:
- **Loss train** giảm đều là mô hình đang học. Nếu không giảm: tốc độ học quá lớn/nhỏ, hoặc dữ liệu có vấn đề.
- **Loss validation** đi xuống rồi đi lên trong khi loss train vẫn giảm: dấu hiệu **học vẹt** (overfitting).
  Đường kẻ dọc là epoch được chọn.
- AUC validation gần 1 ngay từ sớm là bình thường với validation nhỏ; nhìn loss sẽ rõ hơn.

In [ ]:
def plot_history(history, title):
    best = history.loc[history["val_loss"].idxmin(), "epoch"]
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
    axes[0].plot(history["epoch"], history["train_loss"], label="train")
    axes[0].plot(history["epoch"], history["val_loss"], label="validation")
    axes[0].set_title("loss (BCE)")
    axes[1].plot(history["epoch"], history["train_acc"], label="train")
    axes[1].plot(history["epoch"], history["val_acc"], label="validation")
    axes[1].set_title("độ chính xác ở ngưỡng 0,5")
    axes[2].plot(history["epoch"], history["val_auc"], color="C2")
    axes[2].set_title("AUC validation")
    for ax in axes:
        ax.axvline(best, color="grey", ls="--", lw=0.8)
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    axes[0].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


for run, info in RUNS.items():
    if info["history"] is not None:  # chế độ chỉ test thì không có lịch sử train
        plot_history(info["history"], run)

## 11. Kết quả trên holdout

Bảng đầu tiên gom mọi lần chạy. Các cột:
- `threshold`: ngưỡng chọn trên validation. `accuracy`, `FRR`, `FAR`, `precision`, `recall`, `F1`: đo trên holdout **ở ngưỡng đó**.
- `AUC`, `FRR@FAR=…`: không phụ thuộc ngưỡng, dùng để so mô hình (mục 8).
- `val_FRR`, `val_FAR`: cùng ngưỡng nhưng đo trên validation. Lệch nhiều so với holdout nghĩa là ngưỡng
  chọn trên validation không "chuyển" tốt sang người nói mới.

Sau đó là các biểu đồ của lần chạy cuối: phân bố điểm, đường ROC/DET, lỗi theo nhóm, và **nghe các lỗi tệ nhất**.
Nghe lỗi là cách nhanh nhất để hiểu mô hình: bỏ sót vì giọng quá nhỏ? báo nhầm vì câu "Hey GP…"?

In [ ]:
results_df = pd.DataFrame(RESULTS)
show_cols = ["run", "best_epoch", "threshold", "accuracy", "FRR", "FAR", "F1", "AUC"] + [
    c for c in results_df.columns if c.startswith("FRR@")
] + ["val_FRR", "val_FAR"] + [c for c in results_df.columns if c.startswith("latency_median")]
display(results_df[[c for c in show_cols if c in results_df.columns]].round(4))

info = RUNS[LAST_RUN]
scored, threshold = info["scored"], info["threshold"]
labels, scores = scored["label"].to_numpy(), scored["score"].to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
bins = np.linspace(0, 1, 41)
for category, color in (("positive", "C0"), ("hard_negative", "C3"), ("ordinary_negative", "C1")):
    part = scored.loc[scored["category"] == category, "score"]
    axes[0].hist(part, bins=bins, alpha=0.6, color=color, label=f"{category} ({len(part)})")
axes[0].axvline(threshold, color="k", ls="--", lw=1, label=f"ngưỡng {threshold:.2f}")
axes[0].set_yscale("log")
axes[0].set_title("phân bố điểm trên holdout")
axes[0].set_xlabel("điểm (xác suất Hey GPT)")
axes[0].legend(fontsize=8)

fpr, tpr, _ = roc_curve(labels, scores)
axes[1].plot(fpr, tpr)
axes[1].plot([0, 1], [0, 1], color="grey", lw=0.6)
axes[1].set_title(f"ROC (AUC = {roc_auc_score(labels, scores):.4f})")
axes[1].set_xlabel("FAR")
axes[1].set_ylabel("1 − FRR")

eps = 1e-4
axes[2].plot(np.maximum(fpr, eps), np.maximum(1 - tpr, eps))
point = binary_report(labels, scores, threshold)
axes[2].scatter([max(point["FAR"], eps)], [max(point["FRR"], eps)], color="k", zorder=3, label="ngưỡng đã chọn")
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].set_title("DET: FRR theo FAR (log)")
axes[2].set_xlabel("FAR")
axes[2].set_ylabel("FRR")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Ma trận nhầm lẫn trên holdout:")
display(pd.DataFrame(
    [[point["TP"], point["FN"]], [point["FP"], point["TN"]]],
    index=["thật: có", "thật: không"], columns=["đoán: có", "đoán: không"],
))
print("Lỗi theo nhóm (người nói / nguồn âm thanh):")
display(group_report(scored, threshold).round(3))

In [ ]:
def listen_to_errors(scored, X, threshold, k=3):
    misses = scored[(scored["label"] == 1) & (scored["score"] < threshold)].nsmallest(k, "score")
    alarms = scored[(scored["label"] == 0) & (scored["score"] >= threshold)].nlargest(k, "score")
    for title, rows in (("BỎ SÓT tệ nhất (có Hey GPT, điểm thấp nhất)", misses), ("BÁO NHẦM tệ nhất (không có, điểm cao nhất)", alarms)):
        print(f"\n--- {title}: {len(rows)} file ---")
        for i, row in rows.iterrows():
            print(f"{row['name']}  | nhóm {row['group']} | điểm {row['score']:.3f}")
            display(Audio(X[i] / 32768.0, rate=SAMPLE_RATE))


listen_to_errors(scored, X_hold, threshold)

## 12. Nhìn vào bên trong: XLS-R dựa vào lớp nào, và nghe thấy gì ở đâu?

**Trọng số các lớp.** Đầu phân loại trộn 25 tầng của XLS-R với trọng số học được (mục 6.3). Biểu đồ đầu cho biết mô hình
dựa vào tầng nào nhiều nhất. Tầng 0 là đầu ra của bộ mã hoá CNN (gần âm thanh thô), tầng càng cao càng trừu tượng.
Thường thấy các tầng giữa được dùng nhiều, vì chúng giữ thông tin về âm vị.

**Điểm từng khung.** Như BC-ResNet, đầu phân loại là tuyến tính sau phép lấy trung bình, nên logit = trung bình logit của
124 khung 20 ms. Hình thứ hai vẽ dạng sóng của bốn đoạn holdout (phát hiện đúng, bỏ sót, báo nhầm, loại đúng) kèm đường
điểm từng khung. Đường điểm vọt lên ở đâu là mô hình "nghe thấy" Hey GPT ở đó.

In [ ]:
net = model.float().eval()
weights = torch.softmax(net.inner.layer_weights.detach().float().cpu(), 0).numpy() if net.inner.layer_weights is not None else None
if weights is not None:
    fig, ax = plt.subplots(figsize=(12, 2.8))
    top = net.inner.backbone.config.num_hidden_layers - (net.config.trainable_top_layers if net.config.finetune_mode == "partial" else 0)
    colors = ["C1" if (net.config.finetune_mode == "full" or i > top) and i > 0 else "C0" for i in range(len(weights))]
    ax.bar(range(len(weights)), weights, color=colors)
    ax.set_xticks(range(len(weights)))
    ax.axhline(1 / len(weights), color="grey", ls="--", lw=0.8, label="trọng số đều")
    ax.set_xlabel("tầng (0 = đầu ra CNN; màu cam = lớp được tinh chỉnh)")
    ax.set_ylabel("trọng số")
    ax.set_title("đầu phân loại dựa vào tầng nào của XLS-R")
    ax.legend()
    plt.tight_layout()
    plt.show()


def pick(rows, by, largest):
    if rows.empty:
        return None
    return (rows.nlargest(1, by) if largest else rows.nsmallest(1, by)).index[0]


info = RUNS[LAST_RUN]
scored, threshold = info["scored"], info["threshold"]
positive, negative = scored["label"] == 1, scored["label"] == 0
hit = scored["score"] >= threshold
cases = [
    ("positive phát hiện đúng", pick(scored[positive & hit], "score", True)),
    ("positive BỊ BỎ SÓT", pick(scored[positive & ~hit], "score", False)),
    ("negative BỊ BÁO NHẦM", pick(scored[negative & hit], "score", True)),
    ("negative loại đúng", pick(scored[negative & ~hit & (scored["category"] == "hard_negative")], "score", False)),
]
cases = [(title, i) for title, i in cases if i is not None]
device = next(net.parameters()).device
fig, axes = plt.subplots(len(cases), 1, figsize=(14, 2.4 * len(cases)), squeeze=False)
t = np.arange(NUM_SAMPLES) / SAMPLE_RATE
for ax, (title, i) in zip(axes[:, 0], cases):
    wave = torch.from_numpy(X_hold[i][None]).float().div_(32768.0).to(device)
    frames = net.frame_scores(wave)[0].float().cpu().numpy()
    ax.plot(t, X_hold[i] / 32768.0, lw=0.4, color="grey")
    ax.set_ylim(-1, 1)
    ax.set_ylabel("biên độ")
    ax.set_title(f"{title} | {scored.loc[i, 'name']} | nhóm {scored.loc[i, 'group']} | điểm {scored.loc[i, 'score']:.3f}", fontsize=9)
    twin = ax.twinx()
    times = (np.arange(len(frames)) + 0.5) * CONFIG["window_seconds"] / len(frames)
    twin.plot(times, frames, color="C3", lw=1.5)
    twin.axhline(0, color="C3", lw=0.6, ls="--")
    twin.set_ylabel("logit khung", color="C3")
axes[-1, 0].set_xlabel("giây")
plt.tight_layout()
plt.show()
for title, i in cases:
    print(f"{title}: {scored.loc[i, 'name']}")
    display(Audio(X_hold[i] / 32768.0, rate=SAMPLE_RATE))

## 13. Đánh đổi độ chính xác – tốc độ

Mỗi lần chạy ở mục 9 đã được test trên CPU, batch 1, nên có cùng lúc **độ chính xác** và **thời gian suy luận**.
Mục này đặt chúng cạnh nhau:

- **Bảng:** số tham số, dung lượng (MB, float32), số phép nhân-cộng cho một cửa sổ (MMAC = triệu MAC), AUC, F1,
  FRR khi FAR = 1%, độ trễ trung vị và p90 (ms/clip), và **hệ số thời gian thực** (RTF = độ trễ ÷ 2,5 s;
  nhỏ hơn 1 là nhanh hơn thời gian thực). Khi streaming với bước 0,25 s, độ trễ phải nhỏ hơn 250 ms, và nên nhỏ hơn
  nhiều để máy còn làm việc khác.
- **Biểu đồ trái:** mỗi chấm là một lần chạy. Trục ngang là độ trễ 1 luồng, trục dọc là FRR khi FAR = 1%
  (càng thấp càng tốt), chấm càng to càng nhiều tham số. Mô hình tốt nằm ở **góc dưới bên trái**.
- **Biểu đồ phải:** phân bố thời gian của từng clip ở lần chạy cuối. Đuôi dài bên phải (p99 lớn) nghĩa là thỉnh thoảng
  có lần chạy rất chậm, đáng lưu ý cho hệ thống chạy thời gian thực.

**MAC ít hơn chưa chắc chạy nhanh hơn tương ứng.** Với mô hình rất nhỏ, ở batch 1 phần lớn thời gian là chi phí gọi
từng lớp của PyTorch (cỡ vài chục micro-giây mỗi lớp), không phải phép tính. Khi thử, SincConv có MAC ít hơn M5 khoảng
4 lần nhưng chỉ nhanh hơn chút ít. Vì vậy phải **đo thật** thay vì suy từ MAC. Khi đưa lên thiết bị thật (ONNX, TorchScript,
lượng tử hoá) tỉ lệ này có thể khác.

**So nhiều mô hình trên một biểu đồ.** Ô dưới gom mọi file `*_summary.csv` tìm thấy trong `outputs/` và trong
`/kaggle/input`. Trên Kaggle: mở notebook của mô hình này, *Add Input → Notebook Output* và chọn notebook của mô hình kia
(đã chạy xong), rồi chạy lại ô dưới. Chỉ so độ trễ giữa các dòng có **cùng cột `cpu`**: số đo trên hai CPU khác nhau
không so được. Số cho báo cáo nên đo lại trên laptop bằng `checkpoints_to_test` (mục 9).

In [ ]:
summary = pd.DataFrame(RESULTS)
summary.to_csv(SUMMARY_FILE, index=False)

frames, own = [summary], SUMMARY_FILE.resolve()
for base in (OUTPUT_DIR, Path("/kaggle/input")):
    if base.exists():
        for path in sorted(base.rglob("*_summary.csv")):
            if path.resolve() != own:
                frames.append(pd.read_csv(path))
tradeoff = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["run"], keep="first")
tradeoff["MB"] = tradeoff["params"] * 4 / 1e6
columns = ["run", "model", "cpu", "params", "MB", "MMAC", "AUC", "F1", "FRR@FAR=0.01"] + [
    c for c in tradeoff.columns if c.startswith(("latency_median", "latency_p90", "rtf_"))
]
display(tradeoff[[c for c in columns if c in tradeoff.columns]].round(4))
tradeoff.to_csv(OUTPUT_DIR / "tradeoff.csv", index=False)
if tradeoff["cpu"].nunique() > 1:
    print("Lưu ý: bảng có số đo từ nhiều CPU khác nhau; chỉ so độ trễ giữa các dòng cùng CPU.")

if "latency_median_ms_1t" in tradeoff.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    ax = axes[0]
    for name, part in tradeoff.groupby("model"):
        ax.scatter(part["latency_median_ms_1t"], part["FRR@FAR=0.01"], s=20 + part["params"] / 5_000, alpha=0.7, label=name)
        for _, row in part.iterrows():
            ax.annotate(row["run"], (row["latency_median_ms_1t"], row["FRR@FAR=0.01"]), fontsize=7,
                        xytext=(4, 4), textcoords="offset points")
    spread = tradeoff["latency_median_ms_1t"].max() / tradeoff["latency_median_ms_1t"].min()
    if spread > 10:  # chỉ dùng thang log khi các mô hình chênh nhau nhiều
        ax.set_xscale("log")
    ax.set_xlabel("độ trễ CPU trung vị, 1 luồng, batch 1 (ms)" + (", thang log" if spread > 10 else ""))
    ax.set_ylabel("FRR khi FAR = 1% (thấp hơn là tốt)")
    ax.set_title("đánh đổi: tốt nhất ở góc dưới bên trái")
    ax.legend(fontsize=8)
    ax = axes[1]
    last = RUNS[LAST_RUN]["scored"]
    for col in [c for c in last.columns if c.startswith("latency_ms_")]:
        ax.hist(last[col].dropna(), bins=40, alpha=0.6, label=col.removeprefix("latency_ms_").replace("t", " luồng"))
    ax.set_xlabel("ms mỗi clip")
    ax.set_title(f"thời gian từng clip holdout ({LAST_RUN})")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Chưa có số đo thời gian: đặt CONFIG['cpu_test'] = True.")

## 14. (Tuỳ chọn) Đo báo động nhầm mỗi giờ trên bản ghi dài

FAR theo clip chưa nói được trải nghiệm thật. Khi chạy liên tục, hệ thống cắt cửa sổ 2,5 s **mỗi** `stream_hop_seconds`
(mặc định 0,25 s), tức 14 400 lần hỏi mô hình mỗi giờ. Thước đo chuẩn của wake word là **FA/h**: số lần kích hoạt nhầm
trong một giờ âm thanh không có "Hey GPT".

Cách đo trong ô dưới:
1. Đọc từng file dài trong `long_negative_dir` (tự đổi về mono 16 kHz).
2. Trượt cửa sổ 2,5 s với bước `stream_hop_seconds`, chấm điểm từng cửa sổ.
3. Đếm một lần kích hoạt khi điểm ≥ ngưỡng, rồi **im lặng** `refractory_seconds` giây
   (một câu nói làm nhiều cửa sổ liền nhau vượt ngưỡng, nhưng người dùng chỉ thấy một lần bật).
4. FA/h = số lần kích hoạt ÷ tổng số giờ âm thanh.

**Dùng file nào?** Phải là âm thanh mô hình **chưa từng thấy** khi train. Trong repo, ba bản ghi gốc của
negative thường trong holdout (nhóm `en_american_accent`, `en_american_conversation`, `vi_vlog17` trong
`data/negative/raw/`) không được dùng cho train, nên dùng được. Upload chúng thành một dataset Kaggle riêng rồi ghi
đường dẫn vào `CONFIG["long_negative_dir"]`. Ô dưới tự bỏ qua nếu không có.

In [ ]:
def load_long_audio(path):
    rate, data = wavfile.read(path, mmap=True)
    data = np.asarray(data, np.float32)
    if data.ndim == 2:
        data = data.mean(axis=1)
    if np.abs(data).max() > 1.5:  # số nguyên -> [-1, 1]
        data = data / 32768.0
    if rate != SAMPLE_RATE:
        g = math.gcd(int(rate), SAMPLE_RATE)
        data = resample_poly(data, SAMPLE_RATE // g, int(rate) // g).astype(np.float32)
    return data


@torch.no_grad()
def stream_scores(model, audio, hop_seconds, batch_size=256):
    """Điểm của mọi cửa sổ trượt; trả (thời điểm kết thúc cửa sổ, điểm)."""
    hop = max(1, round(hop_seconds * SAMPLE_RATE))
    if len(audio) < NUM_SAMPLES:
        audio = np.pad(audio, (0, NUM_SAMPLES - len(audio)))
    windows = torch.from_numpy(np.ascontiguousarray(audio)).unfold(0, NUM_SAMPLES, hop)
    model.eval()
    scores = []
    for start in range(0, len(windows), batch_size):
        xb = windows[start : start + batch_size].to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            scores.append(torch.sigmoid(model(xb).float()).cpu())
    ends = (np.arange(len(windows)) * hop + NUM_SAMPLES) / SAMPLE_RATE
    return ends, torch.cat(scores).numpy()


def count_activations(ends, scores, threshold, refractory):
    count, last = 0, -math.inf
    for t, s in zip(ends, scores):
        if s >= threshold and t - last >= refractory:
            count, last = count + 1, t
    return count


long_dir = CONFIG["long_negative_dir"]
if long_dir and Path(long_dir).is_dir():
    files = sorted(Path(long_dir).rglob("*.wav"))
    rows = []
    for path in files:
        audio = load_long_audio(path)
        ends, scores = stream_scores(model, audio, CONFIG["stream_hop_seconds"])
        hours = len(audio) / SAMPLE_RATE / 3600
        n = count_activations(ends, scores, threshold, CONFIG["refractory_seconds"])
        rows.append({"file": path.name, "giờ": hours, "kích hoạt nhầm": n, "FA/h": n / hours})
    stream_df = pd.DataFrame(rows)
    display(stream_df.round(3))
    total = stream_df["kích hoạt nhầm"].sum() / stream_df["giờ"].sum()
    print(f"Tổng: {stream_df['giờ'].sum():.2f} giờ, FA/h = {total:.2f} (ngưỡng {threshold:.3f})")
else:
    print("Bỏ qua: chưa đặt CONFIG['long_negative_dir'] hoặc thư mục không tồn tại.")

## 15. Tổng kết và gợi ý thí nghiệm tiếp theo

Ô dưới gom kết quả mọi lần chạy vào `outputs/xlsr_summary.csv`. Khi chạy nhiều seed, bảng thứ hai cho
**trung bình ± độ lệch chuẩn** theo từng phân phối train. Báo cáo nên ghi cả độ lệch chuẩn: chênh lệch giữa hai
mô hình nhỏ hơn độ dao động giữa các seed thì chưa kết luận được mô hình nào tốt hơn.

**Gợi ý thí nghiệm (theo hướng nghiên cứu của nhóm):**
1. **Liều lượng hard negative:** chạy `train_splits = ["H00", "H25", "H50", "H70"]`, `seeds = [0, 1, 2]`.
   So FRR@FAR=1% và FAR trên nhóm `hard_negative` của holdout giữa các phân phối.
2. **So sánh mô hình:** chạy cùng cấu hình cho các notebook M5, SincConv và BC-ResNet, rồi ghép các file `*_summary.csv`.
   So độ chính xác, số tham số, MAC, độ trễ CPU (mục 13).
3. **Thứ tự từ:** Transformer của XLS-R nhìn cả cửa sổ 2,5 s cùng lúc (self-attention), nên có khả năng phân biệt thứ tự từ; nhưng điểm cuối là trung bình mọi khung. Thêm câu kiểu "GPT… hey" vào tập kiểm tra để xem.
4. **Dữ liệu ít:** thử train trên một phần tập train (sửa `load_train` để lấy 25% / 50%) để xem mô hình nào cần ít dữ liệu hơn.

**Nạp lại một mô hình đã train** (ví dụ để chạy demo trên máy):
```python
ckpt = torch.load("outputs/xlsr_H25_seed0/model.pt", map_location="cpu")
model = build_xlsr(XLSRWakeWordConfig(**ckpt["model_config"])); model.load_state_dict(ckpt["state_dict"], strict=False)
model.load_state_dict(ckpt["state_dict"]); model.eval()
threshold = ckpt["threshold"]
```

In [ ]:
results_df = pd.DataFrame(RESULTS)
results_df.to_csv(SUMMARY_FILE, index=False)
metric_cols = ["accuracy", "FRR", "FAR", "AUC"] + [c for c in results_df.columns if c.startswith("FRR@")]
if len(CONFIG["seeds"]) > 1:
    agg = results_df.groupby("split")[metric_cols].agg(["mean", "std"]).round(4)
    display(agg)
print("Đã lưu:", sorted(str(p) for p in OUTPUT_DIR.glob("*")))